In [ ]:
"""
story_to_animation_ltx.py
 
Music-video animation pipeline.
 
Stage 1 — Whisper transcribes the input song (sliding window, ffmpeg+soundfile,
           no librosa/numba dependency).
Stage 2 — OpenAI turns the lyrics into scene_description + action_sequence entries,
           grouped at verse/paragraph level (not per sentence).
Stage 3 — Z-Image (Tongyi-MAI/Z-Image-Turbo) renders one 1280×704 still per scene.
Stage 4 — LTX-2.3 A2VidPipelineTwoStage animates each still with (image + text +
           audio slice) → video.  No new audio is synthesised; the original song
           slice is muxed directly into each clip.  Two seeded variations are
           produced per scene for your manual pick step.
 
Key improvements over the previous version (sourced from the working
story_to_animation.py implementation):
 
  • Pipeline is instantiated via the PYTHON API (constructor + __call__), NOT the
    argparse CLI.  The CLI's --images / --distilled-lora argv encodings have
    already changed between repo revisions and keep breaking.  The Python API is
    documented and stable.
 
  • QuantizationPolicy is imported from ltx_core.quantization via its
    classmethod (QuantizationPolicy.fp8_cast()), which does NOT mmap the 46 GB
    checkpoint — avoiding "Cannot allocate memory" failures on strict-overcommit
    systems.  Falls back gracefully to older import paths.
 
  • Pre-quantized fp8 checkpoints (filename contains "fp8") are detected and
    loaded with quantization=None — applying fp8-cast on top of an already-fp8
    file causes a KeyError on the gate layers' missing input_scale tensors.
 
  • OffloadMode is imported from ltx_pipelines.utils.types with a try/except
    fallback, instead of being assumed present.
 
  • The distilled checkpoint sets DISTILLED_SIGMAS as stage_1_sigmas and uses
    guidance-free MultiModalGuiderParams (cfg_scale=1.0, stg_scale=0.0) — the
    dev checkpoint path uses STG (cfg_scale=3.0, stg_scale=1.0, stg_blocks=[29]).
 
  • call_supported() introspects the pipeline's __call__ signature and passes
    only the kwargs it actually accepts, so minor upstream signature drift
    degrades gracefully instead of crashing.
 
  • torch.no_grad() (not torch.inference_mode()) wraps generation because the
    spatial upsampler's conv3d path raises "Inference tensors cannot be saved for
    backward" under inference_mode on some revisions.
 
  • Asset auto-discovery (find_ltx2_assets) scans the repo's models/ directory
    and HuggingFace cache so you don't have to hard-code every path.
 
  • LoraPathStrengthAndSDOps is only constructed when the NON-distilled (dev)
    checkpoint is used — the distilled checkpoint does not need it.
 
  • ImageConditioningInput is preferred over a plain tuple for the images arg;
    falls back to (path, 0, 1.0) for older revisions.
 
Run inside the LTX-2 repo environment (~/repos/LTX-2) or with ltx-core and
ltx-pipelines pip-installed into the active conda/system environment:
    pip install -e ~/repos/LTX-2/packages/ltx-core
    pip install -e ~/repos/LTX-2/packages/ltx-pipelines
"""
 
import gc
import inspect
import json
import json_repair
import numpy as np
import os
import random
import re
import secrets
import shutil
import soundfile as sf
import subprocess
import tempfile
import time
import torch
import traceback

from xai_sdk import Client as XAIClient
from xai_sdk.chat import system as grok_system, user as grok_user
 
from datetime import datetime
from difflib import SequenceMatcher
from openai import OpenAI
from PIL import Image
from diffusers import ZImagePipeline
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from moviepy.audio.io.AudioFileClip import AudioFileClip
 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XFORMERS_IGNORE_FLASH_VERSION_CHECK"] = "1"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16
 
MAX_SEED    = np.iinfo(np.int32).max
retry_limit = 3

USE_GROK = False
IMAGE_THEME = "3D animation, intimate and beautiful with special effects"
VIDEO_THEME = "Passionate, dramatic, and emotional closeup scenes. Demonic possession, supernatural, demons, paranormal, psychological thriller."
CORRECTIONS = "none"
 

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
LTX_ROOT   = os.path.expanduser("~/repos/LTX-2")
LTX_MODELS = os.path.join(LTX_ROOT, "models")
 
# "4090" → Ada Lovelace, native FP8, 128 GB RAM
# "3090" → Ampere, no native FP8, 64 GB RAM
MACHINE = "4090"
 
_MACHINE_PRESETS = {
    "4090": {
        # bf16 distilled checkpoint + fp8-cast: the correct combo for cpu offload.
        # fp8-scaled-mm is NOT compatible with block-streaming / cpu offload.
        # If you have the pre-quantized fp8 file (ltx-2.3-22b-distilled-fp8.safetensors)
        # the asset discovery below will find it; the code then sets quantization=None
        # automatically (double-quantising a pre-quantized file breaks gate layers).
        "ltx_quantization": "fp8-cast",
        "ltx_offload":      "cpu",
    },
    "3090": {
        # Ampere has no FP8 tensor cores — fp8-cast still works (weight-only cast).
        # 64 GB RAM is tight with cpu offload; switch to "disk" if OOM.
        "ltx_quantization": "fp8-cast",
        "ltx_offload":      "cpu",
    },
}

CONFIG = {
    "openai_api_key": "",
    "openai_model": "gpt-5.4-nano",
    "openai_model_small_reasoning": "gpt-5.4-nano",
    "openai_model_large": "gpt-5.4-nano",
    "hf_token": "",
    "grok_api_key": "",
    "grok_model": "grok-4.3",
    "base_working_dir": "./images",
    "base_video_dir": "./output",
    "audio_files": [
        "/mnt/d/Share/Audio/SoulFire.mp3",    
    ],
    "device": device,
    "dtype": dtype,
    "retry_limit": retry_limit,
    "MAX_SEED": MAX_SEED,
    # ---- LTX-2.3 model paths (auto-discovered if left blank) ----------------
    # Leave blank to use find_ltx2_assets() auto-discovery, or set explicitly.
    "ltx_checkpoint":              "",   # e.g. "ltx-2.3-22b-distilled-1.1.safetensors"
    "ltx_spatial_upsampler":       "",   # e.g. "ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
    "ltx_distilled_lora":          "",   # only needed for the DEV (non-distilled) checkpoint
    "ltx_distilled_lora_strength": 0.8,
    "ltx_gemma_root":              "",   # e.g. "models/gemma-3-12b"
 
    # ---- LTX-2.3 generation settings ----
    "ltx_width": 1280,          # two-stage requires multiples of 64 (stage 1 runs at half res).
    "ltx_height": 704,          # 1280x704 -> stage1 640x352, both /32 -> valid.
    "ltx_fps": 24,
    "ltx_num_inference_steps": 8,   # stage-1 steps; lower = faster.
    "ltx_negative_prompt": (
        "text, letters, subtitles, titles, words, writting, pages with text, screen with text, "
        "shaky, glitchy, low quality, worst quality, deformed, distorted, disfigured, "
        "motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly, "
        "transition, static, watermark, scene change"
    ),
    "ltx_enhance_prompt": False,
 
    # ---- workflow ----
    "ltx_num_variations": 1,    # you make >=2 animations per scene and pick manually.
    "ltx_max_scene_seconds": 5.0,  # matches your existing 22s scene splitter.
    "ltx_min_scene_seconds": 4.0,
}

CONFIG["ltx_checkpoint"]   = os.path.join(LTX_MODELS, "ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors")
#CONFIG["ltx_quantization"] = "fp8-cast"   # casts bf16 to fp8 on the fly, no mmap
CONFIG["ltx_quantization"] = "none"
CONFIG["ltx_offload"]      = "cpu"        # block streaming: only active block in VRAM
CONFIG["ltx_a2v_guidance_scale"] = 1.0
CONFIG["ltx_enhance_prompt"]     = False

# try EROS
CONFIG["ltx_checkpoint"] = os.path.join(
    LTX_MODELS, "ltx-2.3/10Eros_v1.4_bf16.safetensors"
)
CONFIG["ltx_distilled_lora"] = os.path.join(
    LTX_MODELS, "ltx-2.3/ltx-2.3-22b-distilled-lora-1.1_fro90_ceil72_condsafe.safetensors"
)
CONFIG["ltx_distilled_lora_strength"] = 1.0  # condsafe is safe at full strength for I2V
CONFIG["ltx_quantization"] = "fp8-cast"
CONFIG["ltx_offload"] = "cpu"

# Fold the machine preset in.
CONFIG.update(_MACHINE_PRESETS[MACHINE])
 
# Z-Image still resolution matches the LTX target so the conditioning frame
# is not cropped or distorted.
HEIGHT = CONFIG["ltx_height"]
WIDTH  = CONFIG["ltx_width"]
 
outputs_folder = "./temp_outputs/"
os.makedirs(outputs_folder,                  exist_ok=True)
os.makedirs(CONFIG["base_working_dir"],      exist_ok=True)
os.makedirs(CONFIG["base_video_dir"],        exist_ok=True)
 
# =============================================================================
# Prompt example libraries  — paste your full blocks here unchanged.
# =============================================================================
SCENE_DESCRIPTIONS = '''
scene_description: A mushroom spirit. Magical ethereal being. Mossy, moist forest floor, holding Magical golden dusted tulip, holding out and offering tulip to viewer, cinematic, colorful illuminated flowers, bokeh light orbs, mystical expression, zidiusArt
scene_description: A vibrant, extreme neon-soaked scene captured in a dramatic Dutch angle, featuring a charismatic figure in a futuristic astronaut helmet with pink bunny ears. The black visor of the helmet displays a glowing cartoonish smile and two bright, abstract eyes represented by luminous crosses. The character strikes a dynamic combat pose, exuding confidence, with their semi-transparent energy rifles glowing in electric blue and pink, casually held upward, as if ready for action. They wear a rugged, flamboyant pink fur cape over sleek high-tech armor. Comets streak across a fiery cosmic backdrop filled with colorful nebulae, rocky debris, and distant planets. The surreal, chaotic energy of the setting amplifies the playful yet menacing aura of the bunny-eared warrior, standing at the center of an explosive galactic battlefield.
scene_description: The skeleton king is sitting on his silver throne, under a beam of midnight light. Thin black translucent shawl hangs over their face. The skull is weathered, detailed, and slightly visible from under the thin fabric of the shawl. Wearing a black robe with gold trim and a tall black barbed towering crown fitted with a large, glowing purple gemstone. In ancient throne room. Intense. visually impressive, extremely detailed. zidiusArt
scene_description: A monumental stone hand rises from the ground within an overgrown ruin, captured from a slightly low angle with a subtle forward perspective that draws the viewer into the scene. The hand is weathered and ancient, its surface rough and eroded, marked by cracks, chipped edges, and patches of moss and lichen that have slowly overtaken the structure over time. In the open palm rests a small, fragile nest holding a single pale bird. The bird sits calmly, its posture still and composed, its feathers soft yet slightly disheveled, suggesting quiet vulnerability. The nest appears delicate and loosely constructed, with thin, brittle twigs that contrast against the heavy stone beneath it. The stone hand shows signs of gradual decay, with fine particles breaking away from the fingertips and edges, dissolving into dust that drifts slowly through the air. Small fragments lie scattered below, blending into the surrounding ground. The erosion feels natural and controlled, emphasizing the passage of time rather than destruction. Around the base of the structure, vegetation grows in subdued abundance. Plants appear slightly aged and uneven, with leaves showing minor discoloration and subtle decay. Flowers are present but partially wilted, their petals soft and fading in tone, reinforcing the transient nature of life within the scene. In the background, remnants of classical architecture emerge from shadow, including a worn column and partially collapsed stone walls. These structures are softened by time, with surfaces worn smooth and edges softened, integrating seamlessly into the environment. The lighting is cinematic and restrained, inspired by baroque chiaroscuro. A soft, directional light enters from above, illuminating the bird and the center of the hand with a gentle, almost sacred glow, while the surrounding environment remains in deep shadow. The light reveals texture in the stone, moss, and nest, while leaving much of the scene enveloped in darkness. The color palette is muted and cohesive, built around desaturated greens, earthy browns, soft greys, and pale stone tones. Subtle warm highlights appear where light touches the surfaces, while cooler shadows dominate the overall atmosphere. Fine dust particles and organic debris float in the air, catching the light and adding depth and stillness. The composition is balanced and minimal, with a clear focal point on the fragile life resting within a decaying structure. The overall style blends high-end photorealism with subtle painterly softness in texture and atmosphere, forming an elegant vanitas interpretation of protection, fragility, and inevitable decay, where life persists quietly within the slow erosion of time.
scene_description: concept art hidden face, mysterious, hooded, full body, dark enigmatic warrior, standing, storm of falling shards, glowing ethereal giant heavy thick sword in hand, warrior, intricate armor, tattered cloak that billows in the wind, atmosphere is charged with an otherworldly energy, fantasy realism, dramatic lighting, detailed textures to bring the scene to life, digital artwork, illustrative, painterly, matte painting, highly detailed
scene_description: The close-up side profile of a silhouetted figure sitting under a cherry blossom tree. Vibrant pink luminous petals drift in the air. A dense fog obscuring the background except for the immediate area. Emerald dew tipped Grass. They wear a stylish outfit. Bokeh color pop. Striking visual style, extremely detailed. Late evening, golden yellow warmth. zidiusArt
scene_description: mirage split, A dramatic fantasy scene of a winged woman kneeling in shallow water beneath a stormy sky, her posture lowered and reflective, head slightly bowed. Large translucent wings rise from her back, layered like glass feathers, glowing softly from within. Warm golden light shines through the wings, illuminating their delicate structure and scattering feathers into the air. Above her, dark clouds part as sunlight breaks through, casting radiant beams down onto the scene. The water around her reflects the golden light, rippling gently as feathers drift across the surface. Her dress appears light and flowing, blending with mist and water, giving the impression that she is part of the environment itself. Lighting is cinematic and high contrast, combining deep shadows with radiant highlights. Cool blues and greys of the storm contrast with warm gold and amber light from the wings and sun rays. The overall mood feels emotional and symbolic, evoking themes of transformation, fragility, and quiet strength, captured in a moment between darkness and light
scene_description: original character,ghibli inspired,anime style,desaturated tones,figure covered in leaf-like layered fabric, edges sprouting little buds, standing in meadow at forest edge,from_front,looking at viewer,pollen drifting into air,subtle wind motion,overcast sky,minimal composition,strong silhouette,cinematic depth,masterpiece,best quality,absurdres,newest,official art,face is comical sketch with amber ink, amber saber in sheathed back
scene_description: A full-body shot of a solitary warrior standing in the foreground, silhouetted against intense fire and smoke, facing away from the viewer towards a massive dragon looming directly behind them. The dragon is colossal with dark, spiky scales that catch white highlights, glowing red eyes, and an open mouth revealing rows of sharp teeth and internal flames. The warrior holds a long sword upright in their right hand. The composition centers the dragon's head while framing the warrior at the bottom center. The environment is a chaotic battlefield filled with billowing orange and yellow embers rising into a dark blue sky speckled with bokeh light particles. Lighting is dramatic and high-contrast, dominated by warm incandescent glow from the dragon's maw and surrounding fires contrasting sharply with cool blue shadows on the creature's back and the night sky. The visual style is hyper-realistic fantasy art with detailed textures on the dragon's scales and rough ground surface, utilizing volumetric lighting to illuminate dust and smoke particles.
scene_description: wide environmental shot, a colossal human eye embedded in a rocky mountain formation, surface texture blending skin and stone with natural transitions, fog drifting across the terrain, small human figures walking along a narrow path toward the eye for scale, overcast lighting with soft diffusion, subtle reflections in the eye surface, no exaggerated glow, grounded realism, photoreal, sharp environmental detail
scene_description: CONCEPT & MEDIUM: Hyperrealistic cinematic extreme close-up portrait of a menacing cyber-geisha android, dark futuristic and eerie atmosphere with high detail. SUBJECT DESCRIPTION: A haunting female figure with smooth glossy white porcelain-like skin, striking glowing crimson red eyes with intricate mechanical detail, full black glossy lips, and sharp red digital glitch-style markings and patterns running down her cheeks and forehead, subtle water droplets on her face. ACTION & INTERACTION: Staring directly and intensely at the viewer with an unsettling, emotionless gaze, head slightly tilted forward. APPAREL: Ornate cyberpunk headdress and hood in black and red with intricate metallic chains, dangling beads, and circuit-like embellishments framing her face, additional metallic and chain elements visible around her neck. ENVIRONMENT & FOREGROUND: Extreme close-up focusing on the face and headdress, emphasizing the reflective textures of the glossy skin, metallic ornaments, and fabric. BACKGROUND & LIGHTING: Dark, moody setting with dramatic cinematic lighting creating strong highlights on the white face, red accents, and metallic details, deep shadows enhancing the ominous mood, shallow depth of field with atmospheric bokeh
scene_description: Highly detailed woman, her face is made of a silver metal alloy, with gold mascara and gold lipstick, pale icy-blue eyes, blue pixel glitter sparkle, wrapped in silver, radiant white and red pixel bokeh enveloping her, futuristic minimalism combined with darkly romanticism, glossy finish, intricate imagery. The background is a chaotic flurry of white and blue swooping lights.
scene_description: POV close up cinematic fantasy illustration, a dwarven warrior lunges toward the viewer mid attack, and is mid-air. the two-handed warhammer is raised high with both hands about to strike down, with motion blur and flying sparks, massive armored shoulders and thick gauntlets, braided beard and scarred face in extreme battle frenzy, wide eyes and clenched teeth, breath and spit in cold air, helmet rim and plate armor catching hard rim light, gritty steel textures with dents and mud, background a blurred battlefield of smoke and embers, dramatic chiaroscuro, high contrast, 24mm wide lens look, shallow depth of field, ultra detailed, intense energy, high resolution masterpiece quality
scene_description: Duchesse style, digital illustration, The image is a vibrant and dynamic illustration set against a backdrop of a bustling city at night. The central figure is a woman with short blonde hair styled in loose waves, wearing a stylish outfit that combines casual and edgy elements. She is dressed in a turquoise bomber jacket adorned with various patches, including one that reads Bennett's Emporium and another that says Rochester. Underneath the jacket, she wears a white t-shirt with red text, paired with high-waisted pink cargo pants. Her accessories include hoop earrings and a wristwatch. her hands are in her pockets as she leans back with a knee raised and the sole of one shoe resting up against the car.her blue eyes look off to the left The woman is leaning casually against a vintage blue car, which adds to the retro aesthetic of the scene. The car's shiny exterior reflects the surrounding neon lights, enhancing the overall vibrant atmosphere. In the background, the cityscape is filled with tall buildings illuminated by numerous neon signs, creating a lively and energetic ambiance. Notable among these signs is a large, pink neon sign that reads South Beach, with neon palm trees mimicking the surrounding palm trees along the street. suggesting a place of interest or a landmark within the city. The composition of the image is balanced, with the woman positioned centrally, drawing immediate attention. The use of bright colors—turquoise, pink, and blue—against the darker tones of the night sky creates a striking contrast, making the subject stand out vividly. The lighting is dramatic, with highlights on the woman’s face and clothing, emphasizing her features and the details of her outfit. The overall mood of the image is energetic, youthful, and slightly rebellious, capturing a moment of modern urban life with a touch of nostalgia.
scene_description: Ultra-detailed photorealistic night scene in a snowy, dark Whoville back alley. A huge, menacing Grinch (full classic green fur, yellow glowing eyes, sharp-toothed evil grin) lounges arrogantly on an oversized chaotic throne, hundreds of brightly wrapped Christmas presents that are deliberately torn around him, crumpled, half-ripped open, ribbons dangling, boxes crushed and leaning dangerously – absolutely no neat stacks, everything looks freshly looted and messy, some presents even sliding off the throne arms onto the snow. He’s wearing an oversized, tattered royal-blue Santa coat with dirty white fur trim, hanging wide open, sleeves too long and rolled up, coat tails dragging in the snow. Right beside the throne, dominating the foreground, stands one single gigantic burlap sack with “BUZZ” sprayed in rough black graffiti letters across it. The sack is stuffed to bursting and visibly glowing from within – exclusively filled with hundreds of bright, three-dimensional orange lightning-bolt shaped energy sparks (exactly the Civitai BUZZ icon, but physical and crackling, some bolts spilling out the top and floating in the air like electric fireflies. NO presents inside the sack, only pure orange lightning energy. Scattered on the snowy ground around the throne: more ripped-open gift boxes, paper shredded, bows crushed, a few lonely ornaments rolling away – total festive carnage. Gentle snow falling, cold moonlight mixing with faint colorful Christmas lights from distant windows, neon-like orange glow from the BUZZ sparks reflecting on his green fur and the torn wrapping paper. Photorealistic 8K RAW, razor-sharp textures on fur, torn paper, individual snowflakes, glowing sparks, dirty fur and fabric, dramatic rim lighting, high contrast, pure villainous holiday chaos.
scene_description: Highly detailed, professional photography, hauntingly eerie, beautiful. Robotic adult human, Bokeh effects, high-resolution, dappled black with purples tone, falling bokeh shimmer, wearing cartoon alien hoodie with cartoon alien hood pulled up, long hair. Shooting stars bold and swooping, extreme close-up face, flared view. Heavily shadowed face, underlying flare atmospheric, bokeh light floating. Light particles. Mysterious. multiple bokeh made of neon light. the bokeh each have a distinct color, glowing in a neon electric light. Atmospheric lighting. visually impressive, extremely detailed. zidiusArt
scene_description: A surreal and mesmerizing fantasy world where sound has become physical matter: gigantic translucent musical notes drift slowly through a deep twilight sky like bioluminescent jellyfish, each note pulsing with shifting iridescent colors and soft inner light. A lone figure in flowing robes woven from starlight stands on a floating platform made of crystallized sound waves, playing an ancient violin carved from frozen moonlight. The melody causes the notes to dance and temporarily form glowing bridges between distant floating islands that are actually enormous musical instruments  a grand piano the size of a mountain, a harp with strings made of aurora. Warm purple-gold aurora light mixes with cool cosmic blue, hyper-detailed textures, intricate light refractions, dreamlike yet majestic atmosphere, cinematic composition, masterpiece, best quality, ultra-detailed, 8k
scene_description: A close-up, low-angle shot of a monstrous creature with a textured, dark blue and black rocky head featuring sharp horns and pointed ears. The creature's face is illuminated by intense internal red and orange light emanating from its glowing eyes and mouth, revealing rows of jagged teeth. Its body appears to be made of rough, volcanic rock or obsidian, covered in patches of bright red and yellow fire that swirl around the lower section and foreground. The background consists of blurred bokeh lights in shades of deep blue and purple, creating a sense of depth against the fiery subject. The lighting is high-contrast, with strong highlights on the wet-looking surface of the monster's skin and sharp shadows within the crevices of its features. The visual style is hyper-realistic with heavy saturation, emphasizing the contrast between the cool blues of the rock and the warm oranges of the flames.
scene_description: A young girl with long flowing hair stands on lush green grass, facing a cluster of glowing mushrooms emitting soft golden light. Behind her, a small wooden house with a thatched roof is partially obscured by the grass. The landscape features layered atmospheric perspective with distant elements softly blurred and a misty foreground. Style: Japanese anime aesthetics blended with magical realism, rendered with meticulous luminous textures, fluid linework, and layered transparency. Color palette: radiant golds on mushrooms, cool aquamarines on grass and sky, deep blacks on house silhouette. Lighting: soft golden-hour illumination with celestial glow. Special effects: intricate starbursts at horizon, cascading glitter particles around mushrooms, luminous light trails from house windows. Composition: girl centered, mushrooms to right, house behind, grass in foreground.
scene_description: masterpiece, A sweet little girl with black hair is squatting, an adorable girl looking in profile, Studio Ghibli style, semi-realism, in profile to the camera, offering a lollipop to a dragon with its mouth open. The dragon's head is resting on the ground and its tongue is delicately licking the lollipop. The dragon sucks the lollipop with its long, forked tongue. Forked dragon tongue, very creepy dragon, the dragon has a realistic texture with scales and is very detailed. Dragon in profile, dragon profile, its large eyes look tenderly at the girl. The dragon's teeth are long, sharp, and somewhat dirty. The girl's serenity, despite the danger, is admirable. She carries a small backpack on her back, the girl's hair is slightly stirred by the dragon's breath, the camera is slightly far away and shows the dragon's entire head, the girl looks curiously into the creature's mouth, where the girl's feet are growing cheerful hand-drawn flowers.
scene_description: A whimsical cinematic dark atmospheric digital painting, Masterpiece, high quality, Hyper-realistic, hyper-detailed photograph of sexy beautiful skinny Supergirl, she sits gracefully on the roof of a weathered, vintage house in a wheat field. She wears her iconic blue and red costume, featuring the classic 'S' emblem on her chests, with her cape fluttering. She has very long flowing blonde hair with strongly defined muscles. She gazes upward toward a vast starry sky, the Milky Way stretching overhead in luminous detail. it is midnight, but the stars bring a beautiful dark atmosphere and dramatic lighting. The scene is bathed in soft moonlight and starlight, casting long shadows across the grass. Her cape drapes lightly over the hood, fluttering in the night breeze. dramatic lighting, intriguing atmosphere, sharp magnificent details, delicate features, romantic
scene_description: A dark and moody digital illustration shoot from a low angle about a grim reaper sitting on a crescent moon atop a gothic structure overlooking a city at night. the image also shows a detailed depiction of a skeleton figure in a hooded cloak, sitting on the edge of the crescent, illuminated by the warm glow of the moonlight. on the middle of the image, a male figure appears to be a skeleton, with a skeletal face, wearing a long, dark robe with frayed edges. the figure is seated with their hands clasped together in a contemplative pose, with their upper body facing the viewer. their eyes are closed, and their mouth is slightly open, as if they are deep in thought. in the background, a dark, ominous sky is filled with dark clouds, and the city below is lit up with glowing city lights, creating a dramatic and eerie atmosphere. the overall mood is dark and foreboding, evoking a sense of dread and mystery.
scene_description: color comic book pencil sketch., showcasing a blonde naked and barefooted girl crawling on all fours, she s cute and and got her hair in a ponytail, she s wearing a silver collar, she got medium breast slim and slender, she s submissively looking up at the brunette woman. a seductive light brunette woman walks beside her, she s dressed in a slim and short white sundress, and white sneakers, she s holding a leather leash, the leash is attached to the blonde woman s collar. walking through the lush woods. dramatic lighting, sunny, happy
scene_description: A cinematic, photorealistic shot of a lone child in a vibrant red raincoat walking away from the camera down a desolate, flooded city street. The child holds a single black balloon floating slightly above their head. The environment is post-apocalyptic and gritty, with debris scattered on the wet asphalt reflecting the overcast sky. Tall, dilapidated buildings line both sides of the narrow alleyway, creating a sense of claustrophobia and isolation. The lighting is moody and atmospheric, dominated by cool blues and greys that contrast sharply with the striking red of the coat. Rain streaks are visible in the air, adding to the somber mood. High detail, 8k resolution, dramatic composition, depth of field focusing on the child's back
scene_description: A european young woman, reality photo,The late afternoon light spills into a luxurious, marble-tiled shower, illuminating a scene of raw, unsettling intimacy. A young woman, seemingly in her late teens or early twenties, stands with a slightly hunched posture, her face pressed against the cool, damp wall. Her long, wet black hair clings to her shoulders, reflecting the light in shimmering strands. She’s dressed in a simple, white linen garment that clings to her petite, almost fragile frame – revealing a shockingly large, firm breast pressing against the wall. Her expression is a complex mix of intense pleasure and palpable distress. Her eyes are wide, a subtle eye roll suggesting a blend of embarrassment and a desperate, almost frantic need. Her lips are slightly parted, glistening with moisture, and her skin has a flushed, vibrant quality. A slight blush creeps up her neck and cheeks, hinting at the exertion and the overwhelming sensation. The scene is dominated by the dynamic tension of a forceful encounter. She’s clearly struggling against an unseen force, her arms held back, her fingers clawing at the wall. Her body is actively being pinned, the pressure of an unseen presence is visible in the way her muscles tense. A dark, toned male figure is present, though only partially visible – a large, white cock straining against her. The action is intense, a raw, almost violent struggle captured in the moment. The composition focuses intensely on her vulnerability, yet also her defiant desire, in a setting that feels both opulent and deeply intimate. The lighting is carefully crafted, highlighting her contours and the dampness of the environment, adding to the scene’s sensual and slightly disturbing atmosphere.This is a photograph taken with an iPhone camera. Amateur photograph. Candid photograph. Casual photograph. Unfiltered photograph. Unposed photograph. Real life photograph. 2010s. Natural light. Subtle shadows. Taken for Facebook. Taken for Flickr. Taken for Instagram. Taken for OnlyFans. Taken for Reddit. Taken for Snapchat.
scene_description: [SUBJECT & COMPOSITION] A naked girl lying on her back taking a high-angle selfie with her top pulled up, exposing her bare breasts. The aspect ratio is roughly 16:9 vertical. The camera is held directly above her face, looking down in a top-down selfie perspective. The composition places her face in the upper third and her large exposed breasts dominating the central foreground. The frame edges crop the very top of her hair, her shoulders, and her lower abdomen. [CHARACTER / OBJECT DETAILS] One girl with a fair complexion and blonde hair that spreads out messily on the surface beneath her head. She has a closed-mouth smile, blue-grey eyes looking directly at the lens, and prominent pink flushing across her cheeks and nose. She wears a small silver stud earring in her right ear. She is wearing a muted grayish-blue, ribbed, long-sleeved top with a white scalloped lace trim at the neckline. The top is pulled up and bunched tightly across her upper chest, completely exposing her large bare breasts. The skin of her breasts is fair with visible faint blue veins, extensive pink flushing, light tan-pink areolas, prominent protruding nipples, and a cluster of small brown moles on the lower left breast. Below her breasts, a smooth section of her bare upper abdomen is visible. Her arms extend outward out of frame to hold the camera. [ENVIRONMENT & BACKGROUND] The setting is an indoor resting surface, likely a bed. Behind her head and to the left is a soft, rumpled cream-colored pillow or sheet. To the right, beneath her hair, the surface is covered by a beige fabric patterned with dark brown, stylized floral or leaf motifs. [LIGHTING & ATMOSPHERE] The scene is illuminated by soft, diffused frontal lighting, characteristic of ambient indoor light mixed with the glow of a phone screen. The light casts gentle catchlights in her eyes and creates soft, rolling specular highlights on the tip of her nose, her forehead, and the upper curves of her breasts. Shadows are shallow and soft-edged, primarily pooling under the bunched fabric of her shirt and deep in the cleavage between her breasts. [COLOR & PALETTE] The palette is dominated by warm, soft flesh tones and muted pastels. The fair skin introduces prominent rosy pink and reddish flushes against pale peach tones. The grayish-blue of the ribbed shirt provides a cool, desaturated contrast to the warm skin. The background consists of neutral cream, beige, and dark brown accents. [TECHNICAL STYLE & RENDERING] The image is a casual digital photograph, bearing the visual hallmarks of an unedited front-facing smartphone selfie taken indoors. The depth of field is deep enough to keep both her facial features and her breasts in sharp focus, while the background fabric falls slightly out of focus. There is a mild digital grain and slight loss of micro-contrast typical of indoor phone sensors, faithfully capturing unidealized skin texture, faint veins, and color mottling. [PROMPT] A woman lying on her back taking a top-down selfie, her top pulled up exposing her large bare breasts, high-angle selfie perspective, vertical aspect ratio. One woman with blonde hair spread out beneath her head, blue-grey eyes looking at the lens, closed-mouth smile, pink flushing on her cheeks and nose. She wears a muted grayish-blue ribbed long-sleeved top with white lace trim, pulled up and bunched across her upper chest to fully expose bare breasts. The breasts feature fair skin, faint blue veins, pink skin flushing, light tan areolas, protruding nipples, and scattered small brown moles on the lower left breast. Her bare upper abdomen is visible below. She lies on a bed with a rumpled cream pillow to the left and a beige fabric with a dark brown leaf pattern to the right. Soft diffused frontal lighting, gentle catchlights in the eyes, soft highlights on the upper curves of the breasts, shallow shadows in the cleavage. Pale warm skin tones with rosy pink flushes, muted grayish-blue fabric, neutral beige and dark brown background. Casual smartphone selfie photography, unedited skin texture, deep depth of field, mild digital grain, indoor lighting.
scene_description: color cartoon sketch., depicting a beautiful naked woman, with long light blonde wavy hair, she is grabbing a woman's hair with both her hands while looking at her, she is leaning back whilst standing tall and is looking down, with her legs wide spread. in front of her is the back of a kneeling woman, with long black hair, her head buried in the blonde woman's lap. color pencil sketch, sketch lines, drawn on vintage beige background
scene_description: detailed cinematic erotic color pencil style illustration: a cozy hotel bedroom at night lit by soft, dim, romantic lighting. Two beautiful young skinny topless women sit on a rumpled bed, kissing tenderly with closed eyes and touching each other intimately, expressive poses and conveying intimacy. in the background, an adult man is tied to a chair, fully clothed, watching in frustration and disbelief. Warm color palette, soft shadows, rounded character designs, expressive faces, clean linework, pastel highlights, cozy nighttime atmosphere, romantic and cheeky mood, tasteful, sensual, erotic and suggestive, stylized and playful composition.
scene_description: masterpiece, best quality, amazing quality, 4k, 8k, very aesthetic, high resolution, ultra-detailed, absurdres, scenery, photorealistic, 1girl, solo, beautiful, cute, sharp face, long hair, brown hair, low ponytail, wavy hair, loosely tucked bangs, brown eyes, double eyelid, aegyo sal, upturned eyes, turning head, looking at viewer, seductive, blush, smile, closed mouth, nude, (round heavy perky fake natural extremely gigantic breasts with enormous bust and massive volume:1.5), (huge puffy nipples:1.5), pussy, fat mons, curvy, thick build, skinny, impossible body anatomy, squeezing own breasts with elbow, arched back, front, sideboob, cleavage, head tilt, from above, portrait, seducing, wet body, wet hair, dark black shower room, raw light, detailed skin, depth of field, photorealistic details
scene_description: In a softly sunlit apartment bedroom, two young women are just waking up. One woman wears a sexy oversized T-shirt and is bottomless without underwear her vagina clearly visible. The other is topless and wears loose sexy shorts. they are french kissing. their eyes are closed. The topless woman is fingering the other woman, she is masturbating her. they are in love. serene and happy atmosphere. the bed is still unkempt from the night. morning sun reaches through the window into this beautiful female bedroom. The light from the window paints golden patterns across the room. Plants on the windowsill soak in the light, and dust particles float in the air. The anime style here blends crisp cel-shading with natural textures in fabrics, hair, and skin. The atmosphere is one of quiet intimacy and freshness, Subtle details a rumpled sheets, mismatched socks, a steaming cup of tea nearby a enrich the storytelling. The composition focuses on tenderness without drama, and the warm glow emphasizes a gentle new blossoming love.
scene_description: A powerful African sorceress standing in a dramatic pose, wearing an intricate flowing gown made of layered fabric that blends dark earth tones with glowing ember-like orange accents. She has deep, rich skin and a regal expression, with a vibrant orange headwrap wrapped elegantly around her head. In one hand, she holds a tall, ornate staff made of twisted wood and ash, while the other hand conjures swirling flames that arc upward in a crescent shape, breaking into glowing fragments. Her dress appears to dissolve into particles and embers toward the bottom, as if transforming into fire and wind. The environment is softly blurred with warm, smoky tones, creating a mystical atmosphere. Highly detailed textures, cinematic lighting, fantasy style, ultra-realistic, 8k resolution, dramatic composition, glowing particles, magical energy.
scene_description: Pussy, The image captures a blonde scandinavian 18 year old woman with green eyes in a low-angle shot taken from below, capturing her standing with her legs widely spread and hands placed on her hips, which subtly spreads her vulva and anus for visibility. The vagina's opening (introitus) is prominently visible at the center, exhibiting a pale pink color with smooth, slightly glossy texture; its shape appears as a gentle vertical slit, suggesting an "innie" vagina as the introitus is prominently protruding. Flanking the vaginal opening are the labia minora and majora, which are lightly textured and exhibit a natural skin tone. Above the vagina, the pubic hair is visible, with a coarse, dark brown texture that contrasts with the surrounding skin. The anus is not visible at this angle and woman's pose. She is looking down smilling
scene_description: Milonara style, two beautiful naked young women, one with with long, curly, flowing blonde hair, the other with red straight hair. the blonde woman is masturbating the redhead woman, the fingers are inside the vagina of the redhead woman, female masturbation, fingering her. their mouths are open in an expression of ecstasy and passion, their eyes are closed. Their skin is warm and peachy with light orange shading. a large, luminous full moon positioned in the background. The moon is bright white with subtle watercolor-like textures in light green, yellow, and blue hues, suggesting a cosmic or ethereal quality. The background is a dark, textured black, contrasting sharply with the illuminated woman and moon. The overall impression is sensual and mystical, with strong linework and vibrant, yet soft, coloring typical of comic book or fantasy illustration
scene_description: detailed painterly portrait combining expressive brushwork and ink ხაზ accents, warm golden light diffused through heavy blue-black atmosphere, cinematic contrast with analog grain; a drowned wetland lies silent under night, water rippling with faint distortions and submerged vegetation piercing the surface; a lone feminine silhouette stands partially turned, draped in layered fabric that feels suspended in time; thin strands of glowing thread unravel from her form, stretching outward into the air and water, weaving an intricate network that pulses softly like a living system.
scene_description: A woman reclines in languid repose on a plush sofa, her long light brown hair cascading over one shoulder as she smiles directly at the camera with full lips parted to reveal gleaming teeth, both hands lifting an intricately knitted multicolored sweaterâone red, one blueâopen to expose her bare torso and breasts under soft fabric folds that clasp her waist. Her legs are spread wide, revealing a glistening vulva bathed in warm ambient light as the green-gray cushions and deep crimson blankets frame her form from below. The scene is captured from a high angle, emphasizing her relaxed posture and intimate positioning within the cozy interior space, where bright indoor lighting casts soft highlights across her skin and creates subtle shadows that define muscle and contour. A single source of even illumination bathes the entire composition in gentle realism with no harsh contrasts or distractions, enhancing the tactile quality of woven fabric, smooth flesh, and warm tones while preserving a naturalistic depth through shallow focus on her midsection and face, all rendered without logo or watermark to maintain unbroken visual clarity
scene_description: 1girl, a very pretty 19yo Russian porn model, igmodel, rlskn, beautiful, very pretty face, cute face, gorgeous. Focus on face, (large expressive, strikingly rich blue eyes:1.5), stylistic windswept golden blonde hair, flawless tasteful makeup, glossy full lips slightly parted. (Skinny:2), short, hourglass figure, tiny waist, thick toned thighs, flat stomach, fit toned abdomen, extremely huge perfect perky breasts, light dusky pink colored small areolas with erect nipples. Nude vulva, thick labia. Wearing (designer lingerie, strappy thong panties, dark navy blue satin texture, gold buckles, intricate geometric patterns) and a (sheer, translucent:1.2) dark gray overthrow with its front open casually resting on her slender shoulders, soft cinematic lighting, detailed texture of the fabric, professional lighting, dramatic shadows playing on her soft skin, vogue aesthetic, hyper-detailed. Seductive look, flirty look, standing, looking at viewer. Incredibly detailed skin texture, vellus hair, subsurface scattering on skin, authentic human imperfections, no airbrushing, masterpiece, photorealistic.
scene_description: (Huge Powerful Crescent Moon: 1.3), Neon Purple Glow, Cinematic Masterpiece, [Puppy Sitting and Looking at the Moon], Eerily Glowing Turquoise Accents, Realistic Landscape with Drops, Endless Mystical Night Sky, Cosmic Dust, Vivid Chiaroscuro with Deep Inky Shadows, Volumetric Lighting, Surreal Landscape
scene_description: realistic, perspective, 1girl, side locks, half updo, solo, open mouth, skirt, black hair, hair ornament, ass, flower, japanese clothes, barefoot, teeth, pussy, tongue, looking back, indoors, hair flower, black kimono, from behind, mole, vaginal, lips, anus, no panties, floral print on kimono, squatting, sex toy, mole under mouth,looking down at viewer,torogao, female masturbation, object insertion, dildo, vaginal object insertion, tiptoes, black dildo riding, ceiling, from below, hands on batt, blush, pussy juice puddled, saliva drooping from tongue
scene_desciprtion: detailed cinematic comic book art. in a huge detailed princess bedroom in a castle. Elle and Arielle are both lying on a huge princess bed in a castle. Elle has a long blonde braid and a golden collar, Arielle has long brown wavy hair, both are totally nude. Elle is lying on her back with her legs spread wide. Arielle is lying on top of Elle, kissing her neck. eyes closed. dramatic lighting. messy princess room
scene_description: Cinematic fantasy photograph. Dark, dramatic cinematic digital artwork in a dark fantasy style: a powerful armored king stands tall on a devastated battlefield littered with broken swords, shields, and debris, wearing a heavy suit of battle-worn armor and a deep crimson red cloak billowing in a cold wind. He raises a king's crown high in one hand as a symbol of conquest and dominance, posture commanding and unyielding. The sky above is overcast and foreboding, thick storm clouds pressing low, muted light breaking through in sharp beams that carve strong chiaroscuro across armor, fabric, and ground. Desaturated palette with rich blacks, iron grays, and blood-red accents, cinematic lighting, high contrast, volumetric haze, drifting ash and dust, epic scale, ultra-detailed textures, haunting and powerful atmosphere, masterwork dark fantasy composition.
scene_description: medium shot of a brunette woman seated in a dimly lit pottery studio, hair pulled back with loose strands framing the face, intense and contemplative gaze directed at the viewer, wearing unbuckled beige canvas overalls with straps hanging loosely, exposing bare skin on the torso and side breast, white wet clay smears texturing the skin of the face arms and chest, hands clasped together in lap with clay residue, messy artistic process, wet clay vase standing on a pottery wheel to the right, ceramic bowl in the foreground, background filled with blurred shelves of white and brown ceramic vessels, large green monstera leaves hanging from above adding depth, soft diffuse lighting, moody atmosphere, muted earth tones, raw and sensual aesthetic, contrast between smooth skin and rough clay textures, cinematic lighting, portrait photography, earth tones, soft focus, textural contrast, intimate mood, natural light, high resolution.
'''
 
ACTION_SEQUENCES = (
    "action_sequence: The man dances energetically, leaping mid-air with fluid arm swings and quick footwork"
    "action_sequence: The girl skateboarding, repeating the endless spinning and dancing and jumping on a skateboard, with clear movements, full of charm"
    "action_sequence: The man dances flamboyantly, swinging his hips and striking bold poses with dramatic flair"
    "action_sequence: The young man writes intensely, flipping papers and adjusting his glasses with swift, focused movements"
    "action_sequence: The girl dances gracefully, with clear movements, full of charm"
    "action_sequence: A jellyfish dances in the sea"
    "action_sequence: A pretty clown girl with blue skin giggles as she twirls a knife between her fingers."
    "action_sequence: The drummer aggressively hits the snare drum and crashes the cymbal matching the attached audio track, dynamic lighting shifts, fast-paced concert stage smoke, low-angle tracking shot moving around the kit."
)

In [ ]:
# =============================================================================
# Utility / memory
# =============================================================================
def reset_memory(dev):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)
        torch.cuda.reset_accumulated_memory_stats(dev)
        torch.cuda.ipc_collect()
 
 
def safe_json_parse(response_text):
    """Robustly extract and parse a JSON list from an LLM response."""
    cleaned_text = response_text.strip()
    if cleaned_text.startswith("```"):
        lines = cleaned_text.splitlines()
        if len(lines) >= 2:
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines[-1].startswith("```"):
                lines = lines[:-1]
            cleaned_text = "\n".join(lines)
    try:
        data = json_repair.loads(cleaned_text)
        if isinstance(data, list):
            # Flatten one level if the LLM wrapped scenes in an extra list
            if data and isinstance(data[0], list):
                data = [item for sublist in data for item in sublist]
            data = [d for d in data if isinstance(d, dict)]
            if data:
                return data
        elif isinstance(data, dict):
            for key in data:
                if isinstance(data[key], list):
                    inner = data[key]
                    if inner and isinstance(inner[0], list):
                        inner = [item for sublist in inner for item in sublist]
                    inner = [d for d in inner if isinstance(d, dict)]
                    if inner:
                        return inner
            return [data]
    except Exception as e:
        print(f"JSON Repair failed: {e}")
    try:
        match = re.search(r'\[.*\]', response_text, re.DOTALL)
        if match:
            data = json_repair.loads(match.group())
            if isinstance(data, list):
                if data and isinstance(data[0], list):
                    data = [item for sublist in data for item in sublist]
                data = [d for d in data if isinstance(d, dict)]
                return data
    except Exception:
        pass
    return []
 
def _truncate_prompt(prompt: str, max_words: int = 300) -> str:
    """Truncate prompt to avoid Gemma 1024 token limit."""
    words = prompt.split()
    if len(words) > max_words:
        truncated = " ".join(words[:max_words])
        print(f"  [prompt] Truncated from {len(words)} to {max_words} words.")
        return truncated
    return prompt
     
# =============================================================================
# Audio helpers — ffmpeg + soundfile (no librosa, no numba)
# =============================================================================
def load_audio_mono_16k(audio_file: str) -> tuple:
    """Decode any audio format to mono float32 at 16 kHz via ffmpeg+soundfile."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", audio_file,
             "-ar", "16000", "-ac", "1", "-f", "wav", tmp_path],
            check=True, capture_output=True,
        )
        audio, sr = sf.read(tmp_path, dtype="float32", always_2d=False)
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    return audio, sr
 
 
def slice_audio(full_audio_path: str, start_sec: float, duration_sec: float,
                out_path: str) -> str:
    """Cut a time slice from the song and write it as WAV at original quality."""
    audio, sr = sf.read(full_audio_path, dtype="float32", always_2d=True)
    audio = audio.T   # (samples, ch) -> (ch, samples)
    n_ch, n_samp = audio.shape
    start  = int(start_sec * sr)
    end    = min(int((start_sec + duration_sec) * sr), n_samp)
    chunk  = audio[:, start:end]
    needed = int(round(duration_sec * sr))
    if chunk.shape[1] < needed:
        pad   = np.zeros((n_ch, needed - chunk.shape[1]), dtype=chunk.dtype)
        chunk = np.concatenate([chunk, pad], axis=1)
    sf.write(out_path, chunk.T, sr)
    return out_path
 
 
def get_audio_duration(audio_file: str) -> float:
    with AudioFileClip(audio_file) as clip:
        return clip.duration
 
 
# =============================================================================
# Whisper transcription
# =============================================================================
def get_audio_pipe():
    model_id  = "openai/whisper-large-v3-turbo"
    processor = WhisperProcessor.from_pretrained(model_id)
    model     = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    ).to(device)
    return processor, model
 
 
# =============================================================================
# Z-Image still generation
# =============================================================================
def load_zimage_models():
    pipe = ZImagePipeline.from_pretrained(
        "Tongyi-MAI/Z-Image-Turbo",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=False,
    )
    pipe.to("cuda")
    return pipe
 
 
@torch.inference_mode()
def generate_image(pipe: ZImagePipeline, prompt: str, negative_prompt: str,
                   resolution: tuple, seed: int):
    prompt = IMAGE_THEME + ", " + prompt
    height, width = resolution
    if seed == -1:
        seed = secrets.randbits(32)
    image = pipe(
        prompt=prompt+" No text, no titles, no written words, no watermark, no signature, no username, no captions.",
        height=height,
        width=width,
        num_inference_steps=9,
        guidance_scale=0.0,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    return image, seed
 
 
# =============================================================================
# LLM CLIENT MANAGEMENT
# =============================================================================
_openai_client = None
_grok_client = None

def _select_model(use_grok: bool = False, openai_model_override: str = None,
                   grok_model: str = None) -> str:
    if use_grok:
        return CONFIG["grok_model"]
    else:
        return CONFIG["openai_model"]
        
def _get_client(use_grok: bool = False):
    return _get_grok_client() if use_grok else _get_openai_client()

def _get_openai_client():
    global _openai_client
    if _openai_client is None:
        _openai_client = OpenAI(api_key=CONFIG["openai_api_key"])
    return _openai_client


def _get_grok_client():
    global _grok_client
    if _grok_client is None:
        _grok_client = OpenAI(api_key=CONFIG["grok_api_key"], base_url="https://api.x.ai/v1")
    return _grok_client
       
def get_openai_prompt_response(prompt, max_completion_tokens=36000,
                                temperature=0.70, openai_model=None, use_grok=None):
    """Main LLM completion entry point. Handles retries + truncation warnings.

    Prompt-size safety: every prompt is passed through a hard token guard before
    the first call so an oversized context is trimmed (and logged) rather than
    rejected by the API. If the provider STILL reports a token-limit error
    (model ceilings vary, and our estimate is heuristic), we treat that as
    deterministic and SHRINK the prompt for the next attempt instead of
    sleeping and resending the identical payload — which is what previously
    burned five retries and ~60s before failing.

    """
    # Merge the cached prefix into the prompt so it sits at the head of the
    # payload.  Grok's automatic prefix caching then serves this stable block
    # from cache on every subsequent call that shares the same prefix, giving
    # us the cache benefit without needing a separate API parameter.
    if use_grok is None:
        use_grok = USE_GROK
    model = _select_model(use_grok, openai_model)
    client = _get_client(use_grok)

    for retry_count in range(retry_limit):
        try:
            # Grok uses max_tokens; OpenAI uses max_completion_tokens.
            # Passing the wrong key is silently ignored, removing the cap entirely.
            token_key = 'max_tokens' if use_grok else 'max_completion_tokens'
            if use_grok:
                response = client.chat.completions.create(
                    **{token_key: max_completion_tokens},
                    reasoning_effort="none",
                    messages=[
                        {"role": "system", "content": (
                            "You are an expert novelist and editor with deep knowledge of "
                            "literary craft, pacing, character voice, and commercial fiction. "
                            "You write prose that is vivid, emotionally resonant, and commercially appealing. "
                            "You never use em-dashes. You use double spaces between sentences. "
                            "All dialogue uses straight quotation marks."
                        )},
                        {"role": "user", "content": prompt},
                    ],
                    model=model, temperature=temperature,
                )
            else:
                response = client.chat.completions.create(
                    **{token_key: max_completion_tokens},
                    messages=[
                        {"role": "system", "content": (
                            "You are an expert novelist and editor with deep knowledge of "
                            "literary craft, pacing, character voice, and commercial fiction. "
                            "You write prose that is vivid, emotionally resonant, and commercially appealing. "
                            "You never use em-dashes. You use double spaces between sentences. "
                            "All dialogue uses straight quotation marks."
                        )},
                        {"role": "user", "content": prompt},
                    ],
                    model=model, temperature=temperature,
                )                
            finish_reason = response.choices[0].finish_reason
            content = response.choices[0].message.content
            if finish_reason == 'length':
                model_name = "Grok" if use_grok else "OpenAI"
                print(f"WARNING ({model_name}): Response truncated. Got {len(content) if content else 0} chars.")
            if content and len(content.strip()) > 0:
                return content
            else:
                model_name = "Grok" if use_grok else "OpenAI"
                print(f"Empty content on retry {retry_count+1} ({model_name}). Finish: {finish_reason}")
        except Exception as e:
            model_name = "Grok" if use_grok else "OpenAI"
            # Token-limit rejections are deterministic: resending the same payload
            # will fail identically. Shrink hard and retry immediately (no sleep).
            wait_time = min(2 ** (retry_count + 1), 30)
            print(f"Attempt {retry_count+1} error ({model_name}): {e}. Waiting {wait_time}s...")
            time.sleep(wait_time)
    print(f"ERROR: All {retry_limit} attempts failed")
    return ""
 
 
def generate_character_profile(text: str, config: dict):
    print("Generating Character Profile...")
    prompt = (
        f"Analyze these lyrics and create a VISUAL CHARACTER PROFILE for the main hero.\n"
        f"Lyrics: {text[:300]}...\n"
        "Output ONLY a 50-word description of their Age, Gender, Clothing, and Vibe."
    )
    try:
        profile = get_openai_prompt_response(
            prompt, temperature=0.7, use_grok=USE_GROK
        )
        return profile.strip().replace('"', '')
    except Exception:
        return "A resilient hero in a distressed olive hoodie and dark jeans."
 
 
def similarity_ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()
 
 
def create_scenes(text: str, character_profile: str, config: dict) -> list[dict]:
    """
    Generate a list of cinematic scenes from song lyrics.
    
    Creates visually stunning, emotionally coherent scenes that form a cohesive
    short film narrative, emphasizing creativity, diversity, and emotional journey.
    """
    prompt = f"""
### ROLE
You are a visionary director of photography and master storyteller creating a short animated film for a song. 
Your work is highly creative, imaginative, fantastical, and emotionally resonant. 
You craft breathtaking visuals that elevate lyrics, music into a powerful visual work of elegant art.

Characters can be humans, animals, anthropomorphic beings, mythical creatures, spirits, or abstract entities.

### CORE OBJECTIVES
- Translate the song into a **cohesive visual story** with a clear emotional arc.
- Take the viewer on a profound emotional journey that mirrors the song's mood progression.
- Create **fantastical, imaginative, and original** scenes — avoid clichés and generic tropes.
- Use rich **emotional symbology**, metaphor, and cinematic poetry rather than literal interpretations.
- Do not include lyrics, text, titles, subtitles, or any form of words to visualize in the scene_description.
- Use rich sensory details, lighting, atmosphere, and emotional tone.  
- Apply and use artistic techniques for composition, light, color, emotions, etc.


### VISUAL STYLE & GUIDELINES
1. **Balance**: Mix intimate character-focused shots (close-ups, emotional intensity) with grand atmospheric/environmental shots (awe, wonder, scale).
2. **Diversity**: Radically vary locations, times of day, weather, color palettes, and visual styles between scenes. 
   Draw from surreal, dreamlike, magical realism, and fantastical aesthetics.
3. **Emotional Depth**: Infuse every scene with strong emotion, passion, symbolism, and depth. 
   Use lighting, weather, particle effects, surreal transformations, and artistic dreamlike elements.
4. **No Text in Scene_Descriptions**: Never include readable text, titles, signs, or lyrics in any scene description.
5. **Cinematic Quality**: Each scene must feel like a high-budget, artistic short film shot — intricate details, dramatic lighting, beautiful composition.
- Use rich sensory details, lighting, atmosphere, and emotional tone.  Apply and use artistic techniques for composition, light, color, emotions, etc.
- Use weather and environment effects for drama, like a crashing wave on a beach or rocky beach, lightning in clouds, rain and wind, sun beams, dusk and dawn, sunrise and sunset, etc.


### STORYTELLING RULES (CRITICAL)
- Create a cohesive emotional journey across all scenes to create an emotional, passionate cinematic film.
- Each scene should represent **4–22 seconds** of screen time.
- Action sequences should not obstruct the visual quality, no nets or webs that would take up the image space and obstruct the scene.
  Do not mention things like chains, tendrels, webs, nets, etc. in the action sequence.
- When lyrics mention a character, try to include a rich character description with the scene description.

### OUTPUT FORMAT
Return **ONLY** a raw JSON array. No markdown, no explanations, no ```json wrapper.

Each scene object must contain exactly these keys:
- `start` (float): Timestamp in seconds of the first lyric line in this group
- `text` (string): All lyric lines grouped into this scene
- `scene_description` (string): 220+ word highly detailed, vivid visual prompt suitable for image/video generation. 
  Include subject, environment, lighting, mood, symbolism, artistic influences, camera angle, and composition.
- `action_sequence` (string): MAX 100 words. One or two short sentences describing **ONLY** camera movement and subject motion. 
- Richly describe characters in scenes and include details, make them elaborate and beautiful, intricate, incredible.  This includes both horror like demons and monsters, and also beautiful things like angels and people.

Describe **sustained cinematic shots** (4–22 seconds), not rapid cuts.

### REFERENCE EXAMPLES
Scene description style:
{SCENE_DESCRIPTIONS}

Action sequence style:
{ACTION_SEQUENCES}

### INPUT DATA
- **Theme:** {VIDEO_THEME}
- **Lyrics (apply corrections):** {text}
- **Main Character Profile:** "{character_profile}"
- **Transcription Corrections to make:** {CORRECTIONS}

### TASK
Analyze the lyrics and character profile. Craft a visually spectacular, emotionally powerful sequence of scenes that tells a unified story with passion and emotion. 
Make it fantastical, deeply imaginative, and visually diverse while staying true to the song's emotional core.  
Each scene is individual and should not contain any information about other scenes.  No scene should reference other scenes.
Each scene should be different than the previous ones (different setting, different actions, etc.) but tell a story when put together.

Generate the JSON scene list now.
"""

    print("Requesting scenes from OpenAI...")
    result = get_openai_prompt_response(
        prompt, temperature=0.85, use_grok=USE_GROK
    )

    if not result:
        print("ERROR: OpenAI returned an empty response.")
        return []

    parsed_scenes = safe_json_parse(result)
    if not parsed_scenes:
        print("ERROR: Failed to parse scenes as JSON.")
        print("--- RAW OUTPUT ---")
        print(result)
        print("--- END ---")
        return []

    print(f"Successfully parsed {len(parsed_scenes)} scenes.")
    return parsed_scenes
 
 
def revise_scenes(scenes: list[dict], character_profile: str, config: dict) -> list[dict]:
    """
    Revise scene descriptions and action sequences with strong emphasis on diversity,
    narrative flow, and elimination of repetition.

    Preserves original timing ('start') and lyric text while enriching visuals.

    Merging is done by an explicit "index" field the model echoes back, rather than
    positional order. This means partial revisions are kept even if the model drops,
    duplicates, or reorders scenes in its response (instead of discarding everything
    just because the returned count doesn't match).
    """
    if not scenes:
        return scenes

    # Simplified input for LLM — index is included so we can safely re-align the
    # response even if the returned list is shorter, longer, or reordered.
    simplified = [
        {
            "index": i,
            "scene_description": s.get("scene_description", ""),
            "action_sequence": s.get("action_sequence", ""),
        }
        for i, s in enumerate(scenes)
    ]

    prompt = f"""
You are a master cinematic video prompt engineer and editor for a short animated film. Your task is to **radically enrich and diversify** a sequence of scenes while maintaining strong character consistency and narrative flow.
Do not remove or combine any scenes; only revise them independently, individually.

**Overall Video Theme:**
{VIDEO_THEME}

**Style Guidelines:**
- Scene descriptions must be highly vivid, imaginative, fantastic, wonderful, detailed, amazing, cinematic, and emotionally engaging (~350 words).
- Do not include lyrics, text, titles, subtitles, or any form of words to visualize in the scene_description.
- Apply transcription corrections where needed: {CORRECTIONS}

**DIVERSITY & ANTI-REPETITION RULES (CRITICAL - ENFORCE STRICTLY):**
- Eliminate ALL repetitive settings, environments, visual motifs, and compositions.
- Never repeat the same location, architecture, machine aesthetic, or ritual more than once.
- Radically vary: location, time of day, weather, color palette, scale, and emotional energy for EVERY consecutive scene.
- Actively transform or replace recurring motifs across scenes.
- Suggested diverse environments: surreal dreamscapes, natural mystical landscapes, abstract emotional realms, memory fragments, futuristic voids, underwater sequences, cosmic scales, intimate domestic spaces, stormy cliffs, cherry blossom realms, abandoned ruins, vibrant living markets, quiet snowy villages, aerial cloud cities, etc.
- Each scene is individual and should not contain any information about other scenes. No scene should reference other scenes.

**Narrative & Emotional Flow:**
- Create a cohesive emotional journey across all scenes to create an emotional, passionate cinematic film.
- Ensure scenes build a clear emotional journey that progresses, changes along with the song.
- Create visual and emotional contrast between consecutive scenes aligned with the song.
- When lyrics mention a character, include a rich character description with the scene description.

**Action Sequence Rules:**
- Keep every action_sequence concise and descriptive, be specific but not overc ontstrained (max 2 sentences, less than 120 words).
- Describe ONLY camera movement and subject motion, do not have conflicting directions.
- NEVER describe appearance, lighting, environment, or scene content as these are in the image or scene descriptions.
- Action sequences should not obstruct the visual quality or character, scene views.  Weather effects like rain, snow, etc. is okay.  Special effects are encouraged.
- Remove any mention of things like chains, tendrels, webs, nets, etc. in the action sequence.

**Reference Examples:**
Scene description style:
{SCENE_DESCRIPTIONS}

Action sequence style:
{ACTION_SEQUENCES}

**Character Profile (maintain appearance, personality, clothing, and behavior exactly):**
{character_profile}

**CRITICAL - INDEX PRESERVATION:**
- Every input scene has an "index" field. Your output objects MUST include that same
  "index" value, unchanged, so revisions can be matched back to the correct scene.
- Do not renumber, skip, or invent index values. Do not add or remove scenes.

**Output Requirements:**
Return ONLY a valid JSON array of objects. Each object must have exactly these keys:
- "index": integer (copied exactly from the matching input scene)
- "scene_description": string (~300 words)
- "action_sequence": string (short motion prompt)

Each scene is individual and should not contain any information about other scenes.  No scene should reference other scenes.
Each scene should be different than the previous ones (different setting, different actions, etc.) but tell a story when put together.
If a scene is similar to a previous scene, radically change it to be different.
Do not combine or remove any scenes, only edit and revise the existing scenes.
Input scenes to revise:
{json.dumps(simplified, indent=2)}
"""

    print("Requesting scene revision from OpenAI...")
    result = get_openai_prompt_response(
        prompt, temperature=0.80, use_grok=USE_GROK
    )

    if not result:
        print("WARNING: OpenAI returned empty response for revision.")
        return scenes

    revised_list = safe_json_parse(result)
    if not revised_list:
        print("WARNING: Revision failed to parse. Keeping originals.")
        return scenes

    if len(revised_list) != len(scenes):
        print(f"NOTE: Model returned {len(revised_list)} scenes for {len(scenes)} "
              f"input scenes. Merging by index instead of position.")

    # Build a lookup from the model's response, keyed by the "index" it echoed back.
    # Invalid/out-of-range/duplicate indices are logged and skipped rather than
    # corrupting the merge.
    revised_by_index: dict[int, dict] = {}
    skipped = 0
    for item in revised_list:
        if not isinstance(item, dict):
            skipped += 1
            continue
        idx = item.get("index")
        if not isinstance(idx, int) or not (0 <= idx < len(scenes)):
            skipped += 1
            continue
        if idx in revised_by_index:
            # Duplicate index in the response; keep the first occurrence.
            skipped += 1
            continue
        revised_by_index[idx] = item

    if skipped:
        print(f"NOTE: Skipped {skipped} malformed/duplicate/out-of-range entries in the response.")

    missing_indices = [i for i in range(len(scenes)) if i not in revised_by_index]
    if missing_indices:
        print(f"NOTE: {len(missing_indices)} scene(s) not revised (kept original): {missing_indices}")

    # Merge with smart change detection + logging
    print(f"Merging {len(revised_by_index)} revised scenes (of {len(scenes)} total)...")
    desc_updates = 0
    action_updates = 0

    for i, original in enumerate(scenes):
        revised = revised_by_index.get(i)
        if revised is None:
            continue  # no valid revision for this scene; original is preserved

        new_desc = revised.get("scene_description", "").strip()
        orig_desc = original.get("scene_description", "").strip()

        if new_desc and new_desc != orig_desc:
            original["scene_description"] = revised["scene_description"]
            desc_updates += 1

        new_action = revised.get("action_sequence", "").strip()
        orig_action = original.get("action_sequence", "").strip()

        if new_action and new_action != orig_action:
            original["action_sequence"] = revised["action_sequence"]
            action_updates += 1

    total_updates = desc_updates + action_updates
    print(f"Successfully applied {total_updates} meaningful changes "
          f"({desc_updates} descriptions + {action_updates} actions) "
          f"across {len(scenes)} scenes.")

    # Quick diversity check
    if len(scenes) > 3:
        first_300 = [s["scene_description"][:300].lower() for s in scenes]
        unique_ratio = len(set(first_300)) / len(first_300)
        if unique_ratio < 0.75:
            print("WARNING: Low scene diversity detected after revision. "
                  "Consider running revision again or increasing temperature.")

    return scenes


# =============================================================================
# Scene duration normalization (min/max enforcement on the JSON scene list)
# =============================================================================
def split_scene_into_segments(total_duration: float, min_seg: float, max_seg: float) -> list[float]:
    """
    Break a duration into a list of segment lengths, each within
    [min_seg, max_seg] seconds. Used both by normalize_scene_durations()
    (JSON-level split) and by process_audio_video_ltx() (render-time
    per-segment chaining).

    max_seg is a HARD technical ceiling — for the render-time call it's the
    longest a single LTX generation can produce — so it must never be
    violated. min_seg is a soft preference (nicer pacing / less
    fragmentation); when the two conflict we back off on min_seg, not
    max_seg. Concretely: we start from the minimum number of segments that
    keeps every piece <= max_seg, then only reduce that count further
    (making pieces longer, to lift them above min_seg) if the next-smaller
    count still keeps every piece <= max_seg. If it can't, we accept a
    piece under min_seg rather than exceed max_seg.

    (Earlier version of this function reduced the segment count down to 1
    whenever that cleared the min_seg check, without re-checking max_seg —
    for any total_duration in (max_seg, 2*min_seg), e.g. 5-8s when
    min=4/max=5, that produced a single segment LONGER than max_seg. At
    render time that oversized segment got silently frame-clamped back
    down to ~max_seg, quietly discarding the rest of that scene's video —
    the root cause of rendered output drifting shorter than the source
    audio.)

    Returns a list of segment durations summing to EXACTLY total_duration.
    """
    total_duration = max(min_seg, total_duration)

    if total_duration <= max_seg:
        return [total_duration]

    n = int(-(-total_duration // max_seg))  # ceiling division, no math import needed

    while n > 1 and (total_duration / n) < min_seg:
        candidate_n = n - 1
        if total_duration / candidate_n > max_seg:
            break  # backing off further would violate the hard max_seg ceiling
        n = candidate_n

    seg_len = total_duration / n
    return [seg_len] * n


def is_render_chainable(duration: float, min_seg: float, max_seg: float) -> bool:
    """
    True if `duration` can be split EXACTLY into some integer number of
    segments each within [min_seg, max_seg] (i.e. there exists an integer n
    with n*min_seg <= duration <= n*max_seg).

    Because max_seg is a hard technical ceiling and min_seg a hard
    technical floor on a single LTX generation, some durations fall in a
    "gap" where no such n exists — e.g. with min=4/max=5, durations in
    (5,8), (10,12), (15,16) are all impossible to chain exactly. A scene
    landing in one of these gaps can't be rendered without either
    exceeding max_seg on one segment (silently frame-clamped -> video
    quietly shorter than its audio) or going under min_seg on one segment
    (silently frame-padded up to the floor -> video quietly longer,
    pushing every later scene's timing out of sync). normalize_scene_durations()
    uses this to keep every scene's final duration out of the gaps
    entirely, rather than letting the render step silently compensate.
    """
    if duration <= 0 or max_seg <= 0:
        return False
    n_min = int(-(-duration // max_seg))  # fewest segments that could cover duration
    n_min = max(n_min, 1)
    return n_min * min_seg <= duration + 1e-9


def _merge_scene_pair(earlier: dict, later: dict) -> dict:
    """
    Combine two adjacent (chronologically ordered) scenes into one.

    - The merged entry starts at `earlier`'s start time, so no audio time
      between the two original scenes is lost.
    - Keeps whichever scene_description is longer (treated as the richer /
      more detailed one), along with the action_sequence that was paired
      with it.
    - Concatenates both scenes' lyric `text` so no transcribed lyrics are
      dropped.
    """
    earlier_desc = (earlier.get("scene_description") or "").strip()
    later_desc = (later.get("scene_description") or "").strip()

    if len(later_desc) > len(earlier_desc):
        kept_description = later.get("scene_description", "")
        kept_action = later.get("action_sequence", "")
    else:
        kept_description = earlier.get("scene_description", "")
        kept_action = earlier.get("action_sequence", "")

    merged_text = " ".join(
        t for t in [
            (earlier.get("text") or "").strip(),
            (later.get("text") or "").strip(),
        ] if t
    )

    merged = dict(earlier)
    merged["start"] = earlier["start"]
    merged["text"] = merged_text
    merged["scene_description"] = kept_description
    merged["action_sequence"] = kept_action
    return merged


def normalize_scene_durations(scenes: list[dict], last_end_value: float,
                               config: dict) -> list[dict]:
    """
    Enforce duration bounds directly on the LLM-authored scene list, BEFORE
    revise_scenes() runs.

    IMPORTANT: ltx_max_scene_seconds (e.g. 5s) is the cap on a single LTX
    *render segment*, not on a whole JSON scene. A JSON scene is rendered
    as a CHAIN of one or more segments (see process_audio_video_ltx), so it
    can legitimately span up to json_scene_hard_cap = ltx_max_scene_seconds
    * 4 + ltx_min_scene_seconds (e.g. 5*4+4 = 24s) without needing to be
    split here at all. We only split a JSON scene once it exceeds that
    total, and when we do, we aim for json_scene_soft_target-sized pieces
    (ltx_max_scene_seconds * 4, e.g. 20s) so as not to over-fragment the
    story — but we'll let a piece grow up to the hard cap instead of
    leaving a remainder under ltx_min_scene_seconds.

    ALSO IMPORTANT: because ltx_min_scene_seconds and ltx_max_scene_seconds
    are both hard technical bounds on a single render segment, some
    durations are simply impossible to chain exactly — e.g. with min=4s/
    max=5s, nothing in (5,8), (10,12), or (15,16) seconds can be split into
    segments that are each within [4,5]. A scene landing in one of these
    gaps would otherwise get silently frame-clamped short (dropping video)
    or frame-padded long (adding video and pushing every later scene's
    timing out of sync) at render time — see is_render_chainable(). This
    function keeps every scene's final duration out of those gaps.

    This makes the scenes list authoritative for render timing: after this
    function (and revise_scenes(), which only edits text fields — never
    'start') runs, no scene's duration should change again. Render code
    (process_audio_video_ltx) must map each scene's exact JSON-defined span
    onto its rendered clip so picture stays in sync with its audio slice.

    - Scenes that are too short OR whose duration isn't exactly
      render-chainable are merged into the NEXT scene (repeatedly, if
      needed, until the combined duration clears both problems). The very
      last scene has no "next" to merge into, so if it's still
      short/unchainable after all other merging, it is folded backward
      into the previous scene instead.
    - Scenes longer than json_scene_hard_cap are split into consecutive
      copies — identical scene_description/action_sequence/text, only
      'start' changes — spaced across the original duration, aiming for
      json_scene_soft_target-sized pieces. revise_scenes() is what
      diversifies the resulting duplicate descriptions, since it already
      revises every scene independently by index; splitting only has to
      produce the copies. Splitting can itself land a piece in a gap, so a
      second render-chainability merge pass runs afterward too.
    """
    if not scenes:
        return scenes

    min_sec = float(config["ltx_min_scene_seconds"])          # floor for a JSON scene AND a render segment
    render_seg_max_sec = float(config["ltx_max_scene_seconds"])  # ceiling for a SINGLE render segment, not a JSON scene

    # A JSON scene is rendered as a chain of 1+ segments, so its own ceiling
    # is much larger than a single segment's. Only split at the JSON level
    # once a scene exceeds the hard cap; when splitting, target the softer,
    # smaller size so we don't produce needlessly long chained scenes.
    json_scene_soft_target = render_seg_max_sec * 4          # e.g. 20s
    json_scene_hard_cap = json_scene_soft_target + min_sec   # e.g. 24s

    def _merge_pass(scene_list, needs_merge):
        """Forward-merge consecutive scenes while needs_merge(duration) is
        True for the current (possibly already-merged) scene; then fold a
        still-failing final entry backward into its predecessor. Loops
        internally so cascading merges (a merge that's still short/
        unchainable) keep pulling in further neighbors automatically."""
        out: list[dict] = []
        idx = 0
        total = len(scene_list)
        while idx < total:
            current = dict(scene_list[idx])
            next_start = scene_list[idx + 1]["start"] if idx + 1 < total else last_end_value
            duration = next_start - current["start"]
            while needs_merge(duration) and idx + 1 < total:
                current = _merge_scene_pair(current, scene_list[idx + 1])
                idx += 1
                next_start = scene_list[idx + 1]["start"] if idx + 1 < total else last_end_value
                duration = next_start - current["start"]
            out.append(current)
            idx += 1
        if len(out) >= 2:
            tail_duration = last_end_value - out[-1]["start"]
            while needs_merge(tail_duration) and len(out) >= 2:
                tail = out.pop()
                out[-1] = _merge_scene_pair(out[-1], tail)
                tail_duration = last_end_value - out[-1]["start"]
        return out

    def _choose_gap_free_pieces(duration):
        """
        Split `duration` (> json_scene_hard_cap) into pieces close to
        json_scene_soft_target such that EVERY piece is individually
        render-chainable (see is_render_chainable). Equal division can
        land exactly on a render-chainability gap (e.g. splitting 31.69s
        into two 15.845s pieces when (15,16)s is unchainable), so this
        tries nearby piece-counts first, then falls back to an uneven
        split -- growing one piece from a safe size until the remainder is
        also chainable, recursing if the remainder still exceeds the hard
        cap.
        """
        base_n = max(2, round(duration / json_scene_soft_target))
        candidates = sorted(range(2, base_n + 6), key=lambda k: abs(k - base_n))
        for cand_n in candidates:
            piece = duration / cand_n
            if piece <= json_scene_hard_cap and is_render_chainable(piece, min_sec, render_seg_max_sec):
                return [piece] * cand_n

        # No equal split in range works -- fall back to an uneven split:
        # peel off one render-chainable, hard-cap-respecting piece and
        # recurse on the remainder.
        step = 0.05
        first = min(json_scene_hard_cap, json_scene_soft_target)
        while first >= min_sec:
            remainder = duration - first
            if is_render_chainable(first, min_sec, render_seg_max_sec):
                if remainder <= 1e-6:
                    return [first]
                if remainder <= json_scene_hard_cap:
                    if is_render_chainable(remainder, min_sec, render_seg_max_sec):
                        return [first, remainder]
                else:
                    return [first] + _choose_gap_free_pieces(remainder)
            first -= step

        # Should be unreachable given render_seg_max_sec > 0, but never
        # crash the pipeline over it -- the post-split chainability merge
        # pass (below) will catch and fix any leftover gap duration.
        return [duration]

    def _split_pass(scene_list):
        """Split any scene above json_scene_hard_cap into consecutive
        duplicate copies, each individually render-chainable."""
        out: list[dict] = []
        for idx, scene in enumerate(scene_list):
            start = scene["start"]
            end = scene_list[idx + 1]["start"] if idx + 1 < len(scene_list) else last_end_value
            duration = end - start
            if duration <= json_scene_hard_cap:
                out.append(scene)
                continue
            piece_lengths = _choose_gap_free_pieces(duration)
            piece_start = start
            for piece_len in piece_lengths:
                piece = dict(scene)   # duplicate; only 'start' differs per copy
                piece["start"] = piece_start
                out.append(piece)
                piece_start += piece_len
        return out

    # A duration below min_sec is automatically render-unchainable too (with
    # n=1, min_sec*1 > duration), so a single chainability check covers both
    # concerns.
    def _needs_merge(duration):
        return not is_render_chainable(duration, min_sec, render_seg_max_sec)

    # Every scene needs a start time before we can reason about durations.
    for i, s in enumerate(scenes):
        if s.get("start") is None:
            s["start"] = 0.0 if i == 0 else scenes[i - 1].get("start", 0.0)

    scenes = sorted(scenes, key=lambda s: s["start"])
    n = len(scenes)

    # ---- PASS 1: merge scenes that are too short or land in a render gap --
    merged = _merge_pass(scenes, _needs_merge)
    print(f"  [normalize] Merge pass: {n} -> {len(merged)} scene(s) "
          f"(min={min_sec}s, render-chainability enforced).")

    # ---- PASS 2: split scenes above json_scene_hard_cap -------------------
    final_scenes = _split_pass(merged)
    print(f"  [normalize] Split pass: {len(merged)} -> {len(final_scenes)} scene(s) "
          f"(only scenes > {json_scene_hard_cap}s were split, targeting "
          f"~{json_scene_soft_target}s pieces).")

    # ---- PASS 3: converge -- a merge can reintroduce a hard-cap violation,
    # and a split can reintroduce a gap duration, so alternate the two
    # passes until the scene list stops changing rather than assuming one
    # round of each is enough (it usually is, but this guarantees it).
    for round_num in range(6):
        after_merge = _merge_pass(final_scenes, _needs_merge)
        after_split = _split_pass(after_merge)
        if len(after_split) == len(final_scenes) and after_split == final_scenes:
            final_scenes = after_split
            break
        if len(after_split) != len(final_scenes):
            print(f"  [normalize] Convergence round {round_num + 1}: "
                  f"{len(final_scenes)} -> {len(after_split)} scene(s).")
        final_scenes = after_split
    else:
        print(f"  [normalize] WARNING: duration normalization did not "
              f"converge after 6 rounds ({len(final_scenes)} scenes) -- "
              f"this should not happen with typical min/max ratios; "
              f"proceeding with the current result.")

    return final_scenes

  
# =============================================================================
# Scene pipeline: transcribe -> merge -> LLM scenes -> split
# =============================================================================
def process_audio_scenes(audio_file: str, config: dict):
    audio_basename = os.path.splitext(os.path.basename(audio_file))[0]
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_id  = f"{audio_basename}_{timestamp}"
 
    audio_images_dir = os.path.join(config["base_working_dir"], unique_id)
    audio_videos_dir = os.path.join(config["base_video_dir"],   unique_id)
    os.makedirs(audio_images_dir, exist_ok=True)
    os.makedirs(audio_videos_dir, exist_ok=True)
 
    # --- STEP 1: load audio (ffmpeg + soundfile, no librosa/numba) -----------
    print("Transcribing audio using Whisper (Sliding Window for Full Song)...")
    processor, whisper_model = get_audio_pipe()
    full_audio, sampling_rate = load_audio_mono_16k(audio_file)
    total_duration = len(full_audio) / sampling_rate
    print(f"Total audio duration: {total_duration:.2f}s")
 
    # --- STEP 2: sliding-window Whisper transcription ------------------------
    segments     = []
    chunk_length = 30
    overlap      = 5
    step         = chunk_length - overlap
 
    for start_sec in range(0, int(total_duration), step):
        end_sec = min(start_sec + chunk_length, total_duration)
        print(f" > Processing chunk: {start_sec}s to {end_sec}s")
        chunk_audio = full_audio[
            int(start_sec * sampling_rate): int(end_sec * sampling_rate)
        ]
        inputs = processor(
            chunk_audio, sampling_rate=16000,
            return_tensors="pt", return_attention_mask=True,
        )
        input_features = inputs.input_features.to("cuda").to(dtype)
        attention_mask = inputs.attention_mask.to("cuda")
        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features,
                attention_mask=attention_mask,
                return_timestamps=True,
                task="transcribe",
                max_new_tokens=444,              # 448 total - 4 special tokens
                no_speech_threshold=0.6,
                logprob_threshold=-1.0,
                compression_ratio_threshold=2.4,
                condition_on_prev_tokens=False,  # prevents hallucination chaining
                temperature=0.0,
            )
        # Use processor.decode (singular) on predicted_ids[0] to avoid the
        # "Boolean value of Tensor with more than one value is ambiguous" error
        # that batch_decode raises when passing a 2D tensor to _decode_with_timestamps.
        chunk_text = processor.decode(
            predicted_ids[0],
            skip_special_tokens=False,
            decode_with_timestamps=True,
        )
        raw_chunks = re.findall(
            r"<\|(\d+\.\d+)\|>(.*?)(?=<\|\d+\.\d+\|>|$)", chunk_text
        )
        for ts_str, txt in raw_chunks:
            clean_txt   = txt.strip()
            relative_ts = float(ts_str)
            if start_sec + chunk_length < total_duration and relative_ts >= step:
                continue
            # Filter hallucinated repeating phrases
            words = clean_txt.split()
            if len(words) > 6:
                trigram = " ".join(words[:3]).lower()
                if clean_txt.lower().count(trigram) > 2:
                    print(f"   > Skipping hallucinated segment at {relative_ts + start_sec:.1f}s")
                    continue
            if clean_txt:
                segments.append((relative_ts + start_sec, clean_txt))
 
    del processor, whisper_model, full_audio
    reset_memory(device)
 
    # --- STEP 3: merge short segments into verse-level chunks ----------------
    segments.sort(key=lambda x: x[0])
    raw_segments = segments.copy()   # keep unmerged copy for retry fallback

    MIN_SEGMENT_SECONDS = float(config["ltx_min_scene_seconds"])
    merged_segments = []
    if segments:
        current_text  = []
        current_start = segments[0][0]
        for idx, (ts, txt) in enumerate(segments):
            current_text.append(txt)
            is_last  = (idx == len(segments) - 1)
            next_ts  = segments[idx + 1][0] if not is_last else total_duration
            span     = next_ts - current_start
            if span >= MIN_SEGMENT_SECONDS or is_last:
                merged_segments.append((current_start, " ".join(current_text)))
                current_start = next_ts
                current_text  = []
        segments = merged_segments
 
    # --- STEP 4: build LLM input text ----------------------------------------
    text_for_llm   = "".join(f"Start: {s:.2f}, Text: {t}\n" for s, t in segments)
    last_end_value = float(total_duration)
    scenes_file_path = os.path.join(audio_images_dir, "scenes.json")
 
    # --- STEP 5: LLM scene generation ----------------------------------------
    char_profile = generate_character_profile(text_for_llm, config)
    print("Requesting initial scenes from LLM...")
    scenes = None
    try:
        scenes = create_scenes(text_for_llm, char_profile, config)
        # Retry with raw unmerged segments if too few scenes returned
        if not scenes or len(scenes) < 5:
            print(f"WARNING: Only {len(scenes) if scenes else 0} scenes returned. "
                  "Retrying with unmerged segments...")
            raw_text = "".join(f"Start: {s:.2f}, Text: {t}\n" for s, t in raw_segments)
            scenes   = create_scenes(raw_text, char_profile, config)
    except Exception as e:
        print(f"LLM Attempt 1 failed: {e}. Retrying...")
        try:
            scenes = create_scenes(text_for_llm, char_profile, config)
        except Exception as e:
            print(f"LLM Critical Failure: {e}")
            return "", audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
    if not scenes:
        return "", audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
    # --- STEP 6: enforce min/max scene duration on the JSON scene list -------
    # Duration bounds are resolved HERE, on the LLM-authored scene list, not
    # at render time. Scenes shorter than ltx_min_scene_seconds are merged
    # into a neighboring scene; scenes longer than ltx_max_scene_seconds are
    # split into consecutive copies. This runs BEFORE revise_scenes() so
    # that (a) revise_scenes can diversify the descriptions of any split
    # duplicates, and (b) no scene's duration changes after this point —
    # from here on, each scene's span drives its render length 1:1, which is
    # what keeps the rendered clips in sync with the audio.
    pre_count = len(scenes)
    scenes = normalize_scene_durations(scenes, last_end_value, config)
    print(f"Duration normalization: {pre_count} -> {len(scenes)} scene(s) "
          f"(min={config['ltx_min_scene_seconds']}s, max={config['ltx_max_scene_seconds']}s).")
 
    # --- STEP 7: revise and diversify -----------------------------------------
    if scenes:
        try:
            print("Revising scene descriptions and enforcing diversity...")
            scenes = revise_scenes(scenes, char_profile, config)
        except Exception as e:
            print(f"Warning: Scene refinement skipped due to error: {e}")
 
    with open(scenes_file_path, "w") as f:
        json.dump(scenes, f, indent=2)
    print(f"Successfully processed {len(scenes)} scenes.")
    return scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
 
# =============================================================================
# Z-Image still generation per scene
# =============================================================================
def process_audio_images(config: dict, scenes, audio_images_dir):
    print("Loading Z-Image pipeline and generating stills...")
    pipe = load_zimage_models()
    resolution      = (HEIGHT, WIDTH)
    negative_prompt = (
        "worst quality, low quality, worst aesthetic, old, blurry, lowres, signature, "
        "artist name, watermark, username, sketch, logo, furry, text, speech bubble, "
        "symbols, words, letteres"
    )
    for image_num, scene in enumerate(scenes, 1):
        description  = scene.get("scene_description") or "an animated scene in a music video"
        image_prompt = description
        print(f"Generating Image {image_num}: {image_prompt[:60]}...")
        image, _ = generate_image(pipe, image_prompt, negative_prompt, resolution, -1)
        image.save(
            os.path.join(audio_images_dir, f"image_{str(image_num).zfill(2)}.jpg"),
            dpi=(300, 300),
        )
    del pipe
    reset_memory(device)
    print(f"VRAM after Z-Image release: "
          f"{torch.cuda.memory_allocated() / 1e9:.1f} GB alloc, "
          f"{torch.cuda.memory_reserved() / 1e9:.1f} GB reserved")
 
 
# =============================================================================
# LTX-2.3 asset discovery
# =============================================================================
def find_ltx2_assets(config: dict) -> dict:
    """
    Locate the LTX-2 model files without importing anything heavy.
    Search order: explicit config paths → models/ subdirs → repo root → HF cache.
    Newest file wins when multiple matches exist.
    """
    repo   = os.path.expanduser(LTX_ROOT)
    result = {
        "checkpoint": "", "checkpoint_is_distilled": False,
        "spatial_upsampler": "", "distilled_lora": "", "gemma_root": "",
        "ready": False, "missing": [],
    }
 
    roots = []
    repo_models = os.path.join(repo, "models")
    if os.path.isdir(repo_models):
        roots.append(repo_models)
    if os.path.isdir(repo):
        roots.append(repo)
    hf_cache = os.path.expanduser(
        os.environ.get("HF_HOME",
            os.environ.get("HUGGINGFACE_HUB_CACHE", "~/.cache/huggingface"))
    )
    if os.path.isdir(hf_cache):
        roots.append(hf_cache)
 
    def _find(explicit: str, *patterns, exclude=()):
        if explicit:
            p = os.path.expanduser(explicit)
            if os.path.exists(p):
                return os.path.abspath(p)
        hits = []
        import fnmatch
        for root in roots:
            for dirpath, _, filenames in os.walk(root):
                for fn in filenames:
                    for pat in patterns:
                        if fnmatch.fnmatch(fn.lower(), pat.lower()):
                            if not any(x in fn.lower() for x in exclude):
                                hits.append(os.path.join(dirpath, fn))
        if not hits:
            return ""
        return max(hits, key=os.path.getmtime)
 
    # Checkpoint: prefer distilled, then dev. On no native FP8, prefer bf16 over fp8.
    explicit_ckpt = config.get("ltx_checkpoint", "")
    result["checkpoint"] = _find(
        explicit_ckpt, "ltx-2*distilled*.safetensors",
        exclude=("lora", "upscaler", "upsampler"),
    )
    if not result["checkpoint"]:
        result["checkpoint"] = _find(
            explicit_ckpt, "ltx-2*dev*.safetensors",
            exclude=("lora", "upscaler", "upsampler"),
        )
    if result["checkpoint"]:
        result["checkpoint_is_distilled"] = (
            "distilled" in os.path.basename(result["checkpoint"]).lower()
        )
    else:
        result["missing"].append("No LTX-2 checkpoint found")
 
    result["spatial_upsampler"] = _find(
        config.get("ltx_spatial_upsampler", ""),
        "*spatial-upscaler-x2*.safetensors", "*spatial-upscaler*.safetensors",
    )
    if not result["spatial_upsampler"]:
        result["missing"].append("Spatial upscaler not found")
 
    result["distilled_lora"] = _find(
        config.get("ltx_distilled_lora", ""),
        "*distilled-lora*.safetensors",
    )
    if not result["distilled_lora"] and not result["checkpoint_is_distilled"]:
        result["missing"].append("Distilled LoRA not found (required with dev checkpoint)")
 
    gemma_explicit = config.get("ltx_gemma_root", "")
    if gemma_explicit and os.path.isdir(os.path.expanduser(gemma_explicit)):
        result["gemma_root"] = os.path.abspath(os.path.expanduser(gemma_explicit))
    else:
        for root in roots:
            for dirpath, _, filenames in os.walk(root):
                if "gemma" in dirpath.lower() and "config.json" in filenames:
                    result["gemma_root"] = dirpath
                    break
            if result["gemma_root"]:
                break
    if not result["gemma_root"]:
        result["missing"].append("Gemma text-encoder directory not found")
 
    result["ready"] = not result["missing"]
    return result
 
 
# =============================================================================
# LTX-2.3 pipeline helpers
# =============================================================================
def call_supported(fn, /, **kwargs):
    """Call fn with only the kwargs its signature accepts (drift tolerance)."""
    sig = inspect.signature(fn)
    if any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()):
        return fn(**kwargs)
    return fn(**{k: v for k, v in kwargs.items() if k in sig.parameters})
 
 
def _build_quantization_policy(quant_str: str, ckpt_path: str):
    ckpt_name = os.path.basename(ckpt_path).lower()
    if "fp8" in ckpt_name:
        print("[LTX2] Pre-quantized FP8 checkpoint — quantization=None.")
        return None

    if not quant_str or quant_str.lower() in ("none", ""):
        return None

    quant_str = quant_str.lower()

    if quant_str == "fp8-cast":
        # Cache the policy so build_policy() only mmaps the checkpoint once
        # per process, not once per pipeline instantiation.
        cache_attr = "_ltx2_fp8_cast_policy_cache"
        cached = globals().get(cache_attr)
        if cached is not None:
            print("[LTX2] fp8-cast policy: using cached policy (no mmap).")
            return cached

        try:
            from ltx_core.quantization.fp8_cast import build_policy
            print("[LTX2] fp8-cast: calling build_policy() — mmaps checkpoint once, then caches.")
            policy = build_policy(ckpt_path)
            globals()[cache_attr] = policy
            print("[LTX2] fp8-cast policy built and cached.")
            return policy
        except Exception as e:
            print(f"[LTX2] build_policy() failed: {e} — running bf16.")
            return None

    print(f"[LTX2] Unknown quantization '{quant_str}' — running bf16.")
    return None
 
 
def _build_offload_mode(offload_str: str):
    """Import OffloadMode with fallback."""
    try:
        from ltx_pipelines.utils.types import OffloadMode
        try:
            return OffloadMode(offload_str)
        except ValueError:
            return OffloadMode[offload_str.upper()]
    except Exception:
        return None
 
 
def _make_image_conditioning(path: str):
    """
    Build the image conditioning input in whatever format this revision accepts.
    Prefers ImageConditioningInput NamedTuple; falls back to plain tuple.
    """
    try:
        from ltx_pipelines.utils.args import ImageConditioningInput
        try:
            return ImageConditioningInput(path, 0, 1.0, 33)  # path, frame, strength, crf
        except TypeError:
            return ImageConditioningInput(path, 0, 1.0)
    except ImportError:
        return (path, 0, 1.0)


def _make_multi_frame_conditioning(frame_paths: list[str], strength: float = 1.0):
    """
    Build a list of image-conditioning inputs anchoring the first
    len(frame_paths) frames of a new generation to an existing sequence of
    real frames — extends a video with actual motion context instead of a
    single static still. Used for every segment after the first in a
    scene's render chain.
    """
    from ltx_pipelines.utils.args import ImageConditioningInput
    conditioning = []
    for idx, path in enumerate(frame_paths):
        try:
            conditioning.append(ImageConditioningInput(path, idx, strength, 33))
        except TypeError:
            conditioning.append(ImageConditioningInput(path, idx, strength))
    return conditioning


def extract_last_n_frames(video_path: str, n: int, out_dir: str, prefix: str) -> list[str]:
    """
    Extract the last `n` frames of a short clip as individual JPEGs, in
    chronological order (true last frame is the final list element).

    Uses os.listdir (no extra imports). Cleans up its scratch files even on
    failure so a crash can't leave hundreds of _scratch_ jpgs behind, and
    logs which source frames were kept so conditioning is verifiable from
    the console output.
    """
    os.makedirs(out_dir, exist_ok=True)
    scratch_tag = f"_{prefix}_scratch_"
    scratch_pattern = os.path.join(out_dir, scratch_tag + "%05d.jpg")

    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", video_path, "-q:v", "2", scratch_pattern],
            check=True, capture_output=True,
        )

        all_frames = sorted(
            os.path.join(out_dir, f)
            for f in os.listdir(out_dir)
            if f.startswith(scratch_tag) and f.endswith(".jpg")
        )
        if not all_frames:
            raise RuntimeError(
                f"extract_last_n_frames: ffmpeg succeeded but no scratch frames "
                f"found for pattern {scratch_tag}*.jpg in {out_dir}"
            )

        keep = all_frames[-n:]
        total = len(all_frames)
        print(f"    [chain] {os.path.basename(video_path)}: {total} frames "
              f"decoded, keeping frames {total - len(keep) + 1}..{total} "
              f"as conditioning ({os.path.basename(keep[0])} .. "
              f"{os.path.basename(keep[-1])})")

        final_paths = []
        for idx, src in enumerate(keep):
            dst = os.path.join(out_dir, f"{prefix}_{idx:03d}.jpg")
            os.replace(src, dst)
            final_paths.append(dst)
        return final_paths
    finally:
        # Remove ALL remaining scratch files for this prefix, success or not.
        for f in os.listdir(out_dir):
            if f.startswith(scratch_tag) and f.endswith(".jpg"):
                try:
                    os.remove(os.path.join(out_dir, f))
                except OSError:
                    pass


def concatenate_clips(segment_paths: list[str], out_path: str, fps: float,
                       overlap_frames: int) -> str:
    """
    Stitch a chain of segment clips (all belonging to the SAME JSON scene)
    into one final video for that scene.

    Every segment after the first was generated with `overlap_frames` of
    real leading frames reconstructing the end of the PREVIOUS segment (for
    motion continuity) — those reconstructed frames must not appear twice
    in the final output. This drops the first `overlap_frames` frames (and
    the matching slice of audio) from every segment after the first, using
    ffmpeg's `concat` FILTER so the trims and timestamp resets happen
    correctly in one frame-accurate pass. Re-encodes (unlike a plain
    stream-copy concat); clips are short so this is fast.
    """
    if len(segment_paths) == 1:
        shutil.copy(segment_paths[0], out_path)
        return out_path

    frame_dur = overlap_frames / fps
    inputs = []
    filter_parts = []
    concat_inputs = []
    for idx, path in enumerate(segment_paths):
        inputs += ["-i", path]
        if idx == 0:
            filter_parts.append(f"[{idx}:v]setpts=PTS-STARTPTS[v{idx}]")
            filter_parts.append(f"[{idx}:a]asetpts=PTS-STARTPTS[a{idx}]")
        else:
            filter_parts.append(
                f"[{idx}:v]trim=start_frame={overlap_frames},setpts=PTS-STARTPTS[v{idx}]"
            )
            filter_parts.append(
                f"[{idx}:a]atrim=start={frame_dur:.6f},asetpts=PTS-STARTPTS[a{idx}]"
            )
        concat_inputs.append(f"[v{idx}][a{idx}]")

    filter_complex = (
        ";".join(filter_parts) + ";" +
        "".join(concat_inputs) +
        f"concat=n={len(segment_paths)}:v=1:a=1[outv][outa]"
    )

    cmd = [
        "ffmpeg", "-y", *inputs,
        "-filter_complex", filter_complex,
        "-map", "[outv]", "-map", "[outa]",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "16",
        "-c:a", "aac", "-b:a", "256k",
        out_path,
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    return out_path

 
def build_ltx_pipeline(assets: dict, config: dict):
    from ltx_pipelines.a2vid_two_stage import A2VidPipelineTwoStage

    ckpt         = assets["checkpoint"]
    upsampler    = assets["spatial_upsampler"]
    gemma        = assets["gemma_root"]
    is_distilled = assets["checkpoint_is_distilled"]
    ckpt_is_fp8  = "fp8" in os.path.basename(ckpt).lower()

    quant = _build_quantization_policy(config.get("ltx_quantization", "fp8-cast"), ckpt)

    # Pre-quantized fp8 checkpoint CANNOT use block streaming (cpu/disk offload).
    # derive_layout() expects input_scale for every param; BF16 gate layers don't
    # have it, causing KeyError: 'attn1.to_gate_logits.input_scale'.
    # Solution: offload_mode=None (full in-VRAM load) for fp8 checkpoints.
    # On a 4090 the fp8 file is ~29.5 GB — it won't fully fit in 24 GB VRAM alone,
    # but the pipeline loads components sequentially so peak VRAM stays manageable.
    if ckpt_is_fp8:
        print("[LTX2] FP8 checkpoint detected — forcing offload_mode=None "
              "(block streaming is incompatible with pre-quantized fp8 checkpoints).")
        offload = _build_offload_mode("none")
    else:
        offload = _build_offload_mode(config.get("ltx_offload", "cpu"))

    # Load distilled LoRA if specified — for 10Eros use the condsafe variant.
    # Unlike standard distilled checkpoint, 10Eros benefits from the LoRA
    # even though it's a distilled-style merge.
    lora_list = []
    if assets.get("distilled_lora"):
        try:
            from ltx_core.loader import LTXV_LORA_COMFY_RENAMING_MAP, LoraPathStrengthAndSDOps
            lora_list = [LoraPathStrengthAndSDOps(
                assets["distilled_lora"],
                float(config.get("ltx_distilled_lora_strength", 1.0)),
                LTXV_LORA_COMFY_RENAMING_MAP,
            )]
            print(f"[LTX2] LoRA loaded: {os.path.basename(assets['distilled_lora'])} "
                  f"@ strength {config.get('ltx_distilled_lora_strength', 1.0)}")
        except Exception as e:
            print(f"[LTX2] WARNING: Could not build LoRA descriptor: {e}")

    pipeline = call_supported(
        A2VidPipelineTwoStage,
        checkpoint_path        = ckpt,
        distilled_lora         = lora_list,
        spatial_upsampler_path = upsampler,
        gemma_root             = gemma,
        loras                  = [],
        quantization           = quant,
        offload_mode           = offload,
    )
    # Reduce VAE peak VRAM — especially helpful with offload_mode=None
    try:
        if hasattr(pipeline, 'video_decoder') and hasattr(pipeline.video_decoder, '_vae'):
            pipeline.video_decoder._vae.enable_slicing()
            pipeline.video_decoder._vae.enable_tiling()
            print("[LTX2] VAE slicing + tiling enabled")
    except Exception as e:
        print(f"[LTX2] VAE slicing not available: {e}")
    return pipeline, is_distilled
 
 
def nearest_8k_plus_1(target_frames: float, min_frames: int, max_frames: int) -> int:
    """LTX requires frame counts of the form 8*k + 1."""
    k = round((target_frames - 1) / 8)
    k = max(k, (min_frames - 1) // 8)
    k = min(k, (max_frames - 1) // 8)
    k = max(k, 1)
    return 8 * k + 1

def generate_scene_clip(pipeline, is_distilled, assets, config,
                        image_path, audio_slice_path, prompt,
                        num_frames, seed, out_path):
    from ltx_core.components.guiders import MultiModalGuiderParams
    from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
    from ltx_pipelines.utils.media_io import encode_video
 
    fps          = float(config["ltx_fps"])
    tiling       = TilingConfig.default()
    video_chunks = get_video_chunks_number(num_frames, tiling)
 
    # image_path is either a single still path (segment 1 of a scene's
    # chain — the scene's own Z-Image still) or a list of frame paths
    # (continuation segments — the last N real frames of the previous
    # segment in the same scene's chain, for motion context).
    images = (
        _make_multi_frame_conditioning(image_path)
        if isinstance(image_path, (list, tuple))
        else [_make_image_conditioning(image_path)]
    )
 
    if is_distilled:
        try:
            from ltx_pipelines.utils.constants import DISTILLED_SIGMAS
            s1_sigmas = DISTILLED_SIGMAS
        except ImportError:
            s1_sigmas = None
        gp = MultiModalGuiderParams(
            cfg_scale=1.0, stg_scale=0.0, rescale_scale=0.0,
            modality_scale=1.0, skip_step=0, stg_blocks=[],
        )
    else:
        s1_sigmas = None
        gp = MultiModalGuiderParams(
            cfg_scale=3.0, stg_scale=1.0, rescale_scale=0.7,
            modality_scale=float(config.get("ltx_a2v_guidance_scale", 3.0)),
            skip_step=0, stg_blocks=[29],
        )
 
    with torch.no_grad():
        video, audio = call_supported(
            pipeline.__call__,
            prompt              = _truncate_prompt(prompt, max_words=300),
            negative_prompt     = _truncate_prompt(config["ltx_negative_prompt"], max_words=150),
            seed                = seed,
            height              = config["ltx_height"],
            width               = config["ltx_width"],
            num_frames          = num_frames,
            frame_rate          = fps,
            num_inference_steps = int(config.get("ltx_num_inference_steps", 8)),
            video_guider_params = gp,
            images              = images,
            tiling_config       = tiling,
            enhance_prompt      = bool(config.get("ltx_enhance_prompt", False)),
            audio_path          = audio_slice_path,
            audio_start_time    = 0.0,
            audio_max_duration  = num_frames / fps,
            stage_1_sigmas      = s1_sigmas,
        )
 
        encode_video(
            video               = video,
            fps                 = fps,
            audio               = audio,
            output_path         = out_path,
            video_chunks_number = video_chunks,
        )
 
    if isinstance(video, torch.Tensor):
        video = video.cpu()
    if isinstance(audio, torch.Tensor):
        audio = audio.cpu()
    del video, audio
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
 
    print(f"  VRAM after cleanup: "
          f"{torch.cuda.memory_allocated()/1e9:.1f} GB alloc, "
          f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")
 
    return out_path
 
 
# =============================================================================
# 3) Full replacement for process_audio_video_ltx()
# =============================================================================
def process_audio_video_ltx(config: dict, scenes, audio_file: str,
                             audio_images_dir: str, audio_videos_dir: str,
                             last_end_value: float, timestamp: str,
                             start_image: int = 0):
    """
    Render each JSON scene as a CHAIN of one or more LTX generations.

    ltx_max_scene_seconds (e.g. 5s) is a hard cap on a SINGLE LTX
    generation, not on a whole JSON scene. normalize_scene_durations()
    already guarantees every scene reaching this function is at most
    ltx_max_scene_seconds*4 + ltx_min_scene_seconds (e.g. 24s), so here we
    split each scene's exact on-screen duration into 1+ segments — each
    within [ltx_min_scene_seconds, ltx_max_scene_seconds] — render them off
    the scene's single Z-Image still, chain each continuation segment onto
    the last real frames of the previous one for motion continuity, and
    concatenate the result into ONE final clip per scene. The segment chain
    always sums to exactly the scene's JSON-defined duration — nothing here
    changes scene durations — so picture stays in sync with its audio slice.
    """
 
    # ---- Diagnostic ----------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"VIDEO PHASE START")
    print(f"  Scenes in list:  {len(scenes)}")
    print(f"  Audio file:      {audio_file}")
    print(f"  last_end_value:  {last_end_value:.2f}s")
    print(f"  start_image:     {start_image}")
 
    existing_images = sorted([
        f for f in os.listdir(audio_images_dir)
        if f.startswith("image_") and f.endswith(".jpg")
    ])
    print(f"  Image files found: {len(existing_images)}")
    if len(existing_images) != len(scenes):
        print(f"  WARNING: image count ({len(existing_images)}) != "
              f"scene count ({len(scenes)}) — mismatch will cause skips!")
    print(f"{'='*60}\n")
 
    # ---- PASS 1: compute per-scene segment chains, write audio slices -------
    audio_slices_dir = os.path.join(audio_videos_dir, "audio_slices")
    os.makedirs(audio_slices_dir, exist_ok=True)
    chain_frames_dir = os.path.join(audio_videos_dir, "chain_frames")
    os.makedirs(chain_frames_dir, exist_ok=True)
    segments_tmp_dir = os.path.join(audio_videos_dir, "_segments_tmp")
    os.makedirs(segments_tmp_dir, exist_ok=True)
 
    keep_temp_segments = bool(config.get("ltx_keep_temp_segments", False))
 
    fps               = float(config["ltx_fps"])
    min_scene_sec     = float(config["ltx_min_scene_seconds"])
    max_scene_sec     = float(config["ltx_max_scene_seconds"])  # per-segment cap
    min_frames        = nearest_8k_plus_1(min_scene_sec * fps, 9, 10_000)
    max_frames        = nearest_8k_plus_1(max_scene_sec * fps, 9, 10_000)
 
    # Real frames of motion context carried from the end of one segment
    # into the start of the next, instead of a single static still.
    # 5 @ 24fps ~= 0.2s. Set ltx_overlap_frames: 1 in config to fall back
    # to old single-frame behavior.
    overlap_frames = max(1, int(config.get("ltx_overlap_frames", 5)))
 
    scene_chains = []
    cumulative_drift = 0.0
    for i, scene in enumerate(scenes):
        current_start = scene.get("start")
        if current_start is None:
            current_start = 0.0 if i == 0 else scenes[i - 1]["start"]
            scene["start"] = current_start
 
        next_start = scenes[i + 1]["start"] if i + 1 < len(scenes) else last_end_value
        duration = next_start - current_start
        if duration <= 0:
            print(f"WARNING: Scene {i+1} has invalid duration ({duration:.2f}s) — "
                  f"forcing {min_scene_sec}s. This should not happen if "
                  f"normalize_scene_durations() ran upstream.")
            duration = min_scene_sec

        # Split THIS scene's exact duration into 1+ render segments, each
        # within [min_scene_sec, max_scene_sec] — the per-generation limit.
        # No duration is dropped or capped here: durations were finalized
        # upstream by normalize_scene_durations(), so segment_lengths
        # always sums back to `duration` exactly.
        segment_lengths = split_scene_into_segments(duration, min_scene_sec, max_scene_sec)
 
        chain = []
        seg_start = current_start   # timeline start of NEW content
        for seg_idx, seg_len in enumerate(segment_lengths):
            overlap_n = overlap_frames if seg_idx > 0 else 0
 
            target_frames = seg_len * fps + overlap_n
            num_frames = nearest_8k_plus_1(
                target_frames, min_frames + overlap_n, max_frames + overlap_n
            )
            new_content_frames   = num_frames - overlap_n
            new_content_duration = new_content_frames / fps
 
            chain.append({
                "seg_start": seg_start,             # timeline start of NEW content
                "overlap_frames": overlap_n,         # frames of context prepended
                "num_frames": num_frames,            # total frames to GENERATE
                "new_content_duration": new_content_duration,
            })
            seg_start += new_content_duration
        scene_chains.append(chain)

        # Drift check: the chain's total new-content duration should match
        # this scene's intended `duration` almost exactly (only sub-frame
        # rounding noise, a few ms). A larger gap here means something
        # upstream produced a segment_lengths entry outside
        # [min_scene_sec, max_scene_sec] and it got silently clamped at the
        # frame-quantization step below -- flag it loudly rather than let
        # it silently shrink the final video.
        chain_total = sum(seg["new_content_duration"] for seg in chain)
        scene_drift = chain_total - duration
        cumulative_drift += scene_drift
        if abs(scene_drift) > 0.15:
            print(f"  [drift] Scene {i+1:02d}: intended={duration:.3f}s "
                  f"chained={chain_total:.3f}s drift={scene_drift:+.3f}s "
                  f"(segments={[f'{s:.2f}' for s in segment_lengths]}) "
                  f"-- running total drift={cumulative_drift:+.3f}s")

    if abs(cumulative_drift) > 1.0:
        print(f"\n  *** WARNING: cumulative render-duration drift is "
              f"{cumulative_drift:+.2f}s across all scenes. The final "
              f"concatenated video will run this much "
              f"{'longer' if cumulative_drift > 0 else 'shorter'} than the "
              f"source audio unless this is investigated. ***\n")
    else:
        print(f"  [drift] Cumulative render-duration drift: "
              f"{cumulative_drift:+.3f}s (negligible).")
 
    total_segments = sum(len(c) for c in scene_chains)
    print(f"Planned {total_segments} chained segment(s) across {len(scenes)} scenes "
          f"({sum(1 for c in scene_chains if len(c) > 1)} scene(s) need >1 segment), "
          f"overlap_frames={overlap_frames} ({overlap_frames/fps:.3f}s).")
 
    print("Writing audio slices for every segment...")
    slice_paths = []
    for i, chain in enumerate(scene_chains):
        seg_paths = []
        for s, seg in enumerate(chain):
            overlap_n  = seg["overlap_frames"]
            num_frames = seg["num_frames"]
            # Continuation segments look back `overlap_n` frames into
            # already-covered audio so audio context matches the real-frame
            # video overlap the segment is conditioned on.
            audio_start = max(0.0, seg["seg_start"] - (overlap_n / fps))
            audio_dur   = num_frames / fps
            slice_path = os.path.join(
                audio_slices_dir, f"scene_{str(i+1).zfill(2)}_seg{str(s+1).zfill(2)}.wav"
            )
            try:
                slice_audio(audio_file, audio_start, audio_dur, slice_path)
                seg_paths.append(slice_path)
                print(f"  Scene {i+1:02d} seg {s+1:02d}: audio_start={audio_start:.2f}s "
                      f"dur={audio_dur:.2f}s frames={num_frames} "
                      f"(overlap={overlap_n}) -> {os.path.basename(slice_path)}")
            except Exception as e:
                print(f"  Scene {i+1:02d} seg {s+1:02d}: FAILED to write audio slice: {e}")
                seg_paths.append(None)
        slice_paths.append(seg_paths)
 
    written = sum(1 for sp in slice_paths for p in sp if p)
    print(f"\nAudio slices written: {written} / {total_segments}\n")
 
    # ---- PASS 2: discover assets and load pipeline ---------------------------
    print("Discovering LTX-2.3 model assets...")
    assets = find_ltx2_assets(config)
    if "10eros" in os.path.basename(assets["checkpoint"]).lower() or \
       "10eros" in config.get("ltx_checkpoint", "").lower():
        assets["checkpoint_is_distilled"] = True
        print("[LTX2] 10Eros detected — treating as distilled checkpoint.")
 
    if not assets["ready"]:
        missing = "\n  - ".join(assets["missing"])
        raise RuntimeError(
            f"LTX-2.3 assets not ready:\n  - {missing}\n\n"
            f"Model files should be in {LTX_MODELS}/ltx-2.3/ and "
            f"{LTX_MODELS}/gemma-3-12b/\n"
            "Required files:\n"
            "  ltx-2.3-22b-distilled-1.1.safetensors  (or the fp8 variant)\n"
            "  ltx-2.3-spatial-upscaler-x2-1.1.safetensors\n"
            "  ltx-2.3-22b-distilled-lora-384-1.1.safetensors\n"
            "  gemma-3-12b/  (google/gemma-3-12b-it-qat-q4_0-unquantized)\n"
            "Install packages once:\n"
            f"  pip install -e {LTX_ROOT}/packages/ltx-core\n"
            f"  pip install -e {LTX_ROOT}/packages/ltx-pipelines"
        )
 
    print(f"[LTX2] Checkpoint:    {os.path.basename(assets['checkpoint'])}")
    print(f"[LTX2] Upsampler:     {os.path.basename(assets['spatial_upsampler'])}")
    print(f"[LTX2] Gemma:         {os.path.basename(assets['gemma_root'])}")
    print(f"[LTX2] DistilledCkpt: {assets['checkpoint_is_distilled']}")
    if assets.get("distilled_lora"):
        print(f"[LTX2] LoRA:          {os.path.basename(assets['distilled_lora'])}")
 
    print("Loading LTX-2.3 A2Vid pipeline (done once, reused for all scenes)...")
    pipeline, is_distilled = build_ltx_pipeline(assets, config)
 
    # ---- PASS 3: render each scene's chain, then assemble ONE final clip ----
    print(f"\nRendering {len(scenes)} scenes x {config['ltx_num_variations']} "
          f"variation(s), {total_segments} segment(s) total...\n")
    for i, scene in enumerate(scenes):
        if i < start_image - 1:
            print(f"Scene {i+1:02d}: skipping (before start_image={start_image})")
            continue
 
        chain     = scene_chains[i]
        seg_paths = slice_paths[i]
 
        image_input = os.path.join(
            audio_images_dir, f"image_{str(i + 1).zfill(2)}.jpg"
        )
        if not os.path.exists(image_input):
            print(f"Scene {i+1:02d}: SKIPPING — still image missing: {image_input}")
            continue
 
        base_action = scene.get("action_sequence") or "an animated image"
 
        for v in range(config["ltx_num_variations"]):
            variation_seed = secrets.randbits(31)
            # Segment 1 conditions on the scene's single Z-Image still.
            # Every later segment conditions on a LIST of the last
            # `overlap_frames` real frames of the previous segment.
            conditioning_image  = image_input
            rendered_segments   = []
            rendered_frame_sets = []   # lists of extracted overlap frames, for cleanup
            chain_ok = True
 
            for s, seg in enumerate(chain):
                audio_slice_path = seg_paths[s]
                if audio_slice_path is None or not os.path.exists(audio_slice_path):
                    print(f"Scene {i+1:02d} seg {s+1:02d} var {v+1}: "
                          f"SKIPPING — audio slice missing")
                    chain_ok = False
                    break
 
                overlap_n  = seg["overlap_frames"]
                num_frames = seg["num_frames"]
                seg_seed   = (variation_seed + s * 104729) % (2 ** 31)
 
                if s == 0:
                    prompt = base_action
                else:
                    prompt = (
                        f"{base_action} Continue the same ongoing camera motion "
                        f"and action smoothly from where it left off — do not "
                        f"restart, reset, or repeat the motion from the beginning."
                    )
 
                seg_name = (f"v_ltx_{str(i+1).zfill(2)}_seg{str(s+1).zfill(2)}"
                            f"_var{v+1}_{timestamp}.mp4")
                seg_path = os.path.join(segments_tmp_dir, seg_name)
                cond_desc = (
                    f"{len(conditioning_image)} overlap frame(s)"
                    if isinstance(conditioning_image, (list, tuple))
                    else os.path.basename(conditioning_image)
                )
                print(
                    f"Scene {i+1:02d} seg {s+1:02d}/{len(chain)} var {v+1}: "
                    f"frames={num_frames} (overlap={overlap_n}) seed={seg_seed} "
                    f"conditioning={cond_desc}"
                )
                try:
                    generate_scene_clip(
                        pipeline, is_distilled, assets, config,
                        conditioning_image, audio_slice_path,
                        prompt, num_frames, seg_seed, seg_path,
                    )
                    print(f"  -> segment rendered: {seg_name}")
                    rendered_segments.append(seg_path)
 
                    if s + 1 < len(chain):
                        next_overlap_n = chain[s + 1]["overlap_frames"]
                        frame_prefix = f"scene_{str(i+1).zfill(2)}_var{v+1}_seg{str(s+1).zfill(2)}"
                        try:
                            frame_paths = extract_last_n_frames(
                                seg_path, next_overlap_n, chain_frames_dir, frame_prefix
                            )
                            conditioning_image = frame_paths
                            rendered_frame_sets.append(frame_paths)
                        except Exception:
                            # A wrong conditioning image silently produces a
                            # broken (resetting) video, which is worse than no
                            # video. Fail the chain loudly instead.
                            traceback.print_exc()
                            print(f"  -> FAILED to extract overlap frames from "
                                  f"{os.path.basename(seg_path)} — aborting this "
                                  f"chain (rendered segments left in "
                                  f"{segments_tmp_dir} for inspection).")
                            chain_ok = False
                            break
                except torch.cuda.OutOfMemoryError:
                    print(f"  -> OOM on scene {i+1} seg {s+1} var {v+1} — "
                          f"flushing and continuing.")
                    gc.collect()
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                    chain_ok = False
                    break
                except Exception:
                    traceback.print_exc()
                    print(f"  -> FAILED: scene {i+1} seg {s+1} var {v+1}")
                    gc.collect()
                    torch.cuda.empty_cache()
                    chain_ok = False
                    break
                time.sleep(1)
 
            # ---- Assemble the final clip for this scene/variation ----------
            final_name = f"v_ltx_{str(i+1).zfill(2)}_var{v+1}_{timestamp}.mp4"
            final_path = os.path.join(audio_videos_dir, final_name)
 
            if not rendered_segments:
                print(f"Scene {i+1:02d} var {v+1}: no segments rendered — "
                      f"skipping final assembly.")
                continue
 
            if not chain_ok:
                print(f"Scene {i+1:02d} var {v+1}: chain incomplete "
                      f"({len(rendered_segments)}/{len(chain)} segment(s) rendered) — "
                      f"leaving partial segment(s) in {segments_tmp_dir} for "
                      f"inspection, no final clip assembled.")
                continue
 
            if len(rendered_segments) == 1:
                shutil.move(rendered_segments[0], final_path)
                print(f"  -> final: {final_name}")
            else:
                try:
                    concatenate_clips(rendered_segments, final_path,
                                       fps=fps, overlap_frames=overlap_frames)
                    print(f"  -> final (chained {len(rendered_segments)} "
                          f"segments): {final_name}")
                    if not keep_temp_segments:
                        for p in rendered_segments:
                            os.remove(p)
                        for frame_paths in rendered_frame_sets:
                            for p in frame_paths:
                                if os.path.exists(p):
                                    os.remove(p)
                except Exception as e:
                    print(f"  -> WARNING: concatenation failed ({e}); leaving "
                          f"{len(rendered_segments)} segment(s) in "
                          f"{segments_tmp_dir} — no final clip assembled.")
 
    del pipeline
    reset_memory(device)
    print(f"\nVideo phase complete. Output: {audio_videos_dir}")

# =============================================================================
# Orchestration
# =============================================================================
def process_all_audios(audio_file, config: dict):
    print(f"Processing audio file: {audio_file}")
    scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp = \
        process_audio_scenes(audio_file, config)
    print(f"{len(scenes)} scenes:\n{json.dumps(scenes, indent=4)}")
    print(f"last_end_value: {last_end_value}  timestamp: {timestamp}")
    process_audio_images(config, scenes, audio_images_dir)
    return config, scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
 
def create_video(config, audio_file):
    config, scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp = \
        process_all_audios(audio_file, config)
    process_audio_video_ltx(
        config, scenes, audio_file,
        audio_images_dir, audio_videos_dir,
        last_end_value, timestamp,
        start_image=0,
    )

In [ ]:
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


In [ ]:
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


In [ ]:
USE_GROK = True

In [ ]:
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


In [ ]:
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


## Run Previous Generation

In [ ]:
# run saved ProphetFist_20251211_192956
timestamp = '20260801_195839'
title = 'ChickenTable'
start_image = 0
scenes_file_path = f'./images/{title}_{timestamp}/scenes.json'
audio_images_dir = f'./images/{title}_{timestamp}'
audio_videos_dir = f'./output/{title}_{timestamp}'

last_end_value = 336.28

with open(scenes_file_path, "r") as scenes_file:
    scenes = json.load(scenes_file)

process_audio_video_ltx(CONFIG, scenes, CONFIG["audio_files"][0], audio_images_dir, audio_videos_dir, last_end_value, timestamp, start_image)